In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import glob
import re
import os

# loading datasets
def load_datasets(data_path):
  datasets = {}
  for file in glob.glob(pathname = data_path + '*.csv'):
    name = file.split('/')[-1].replace('.csv','')
    datasets[name] = pd.read_csv(file)
  return datasets

In [2]:
# all datasets basic profiling
def basic_dataset_profiling(datasets):
  results = []
  for name, dataset in datasets.items():
    row = {
        'dataset': name,
        'rows': dataset.shape[0],
        'columns': dataset.shape[1],
        'duplicates': dataset.duplicated().sum()
    }
    results.append(row)
  return pd.DataFrame(results)

In [3]:
# basic profiling of all columns in all datasets
def basic_columns_profiling(datasets):
  results = []
  for name, dataset in datasets.items():
    for column in dataset.columns:
      row = {
          'dataset': name,
          'column': column,
          'rows': dataset.shape[0],
          'unique': dataset[column].nunique(),
          'duplicates': dataset[column].duplicated().sum(),
          'missing': dataset[column].isna().sum(),
          'dtype': dataset[column].dtype
      }
      results.append(row)
  return pd.DataFrame(results)

In [4]:
# decisions log creation
cleaning_log = []
def cleaning_decisions(
    # dataset - name of the dataset
    # column - name of the column
    # issue - concise description of the issue found
    # evidence - evidence found in profiling
    # action - retained/removed/converted
    # reason - supporting the decision taken
    # before_count - count of the values before changes
    # changes_count - how many values affected
    # after_count - count of the values after changes
    # status - Completed/Profiled

    dataset,
    column,
    issue,
    evidence,
    action,
    reason,
    before_count,
    changes_count,
    after_count,
    status
):
  cleaning_log.append(
      {
          'dataset': dataset,
          'column': column,
          'issue': issue,
          'evidence': evidence,
          'action': action,
          'reason': reason,
          'before_count': before_count,
          'changes_count': changes_count,
          'after_count': after_count,
          'status': status
      }
  )

In [5]:
# cleaning whitespace around object columns
def cleaning_whitespace(datasets):
  for name, dataset in datasets.items():
    object_columns = dataset.select_dtypes(include='object').columns
    for column in object_columns:
      dataset[column] = dataset[column].str.strip()

In [6]:
# cleaning city columns across all the datasets
def cleaning_city(datasets,city_mapping):
  for name, dataset in clean_datasets.items():
    if 'City' in dataset.columns:
        dataset['City'] = (
            dataset['City']
            .replace('', pd.NA)
            .str.title()
            .replace(city_mapping)
        )

In [7]:
# cleaning rating columns in a dataset
def clean_rating(dataset, columns):

    for column in columns:

        invalid = (dataset[column] < 1) | (dataset[column] > 5)

        count = invalid.sum()

        if count > 0:
            print(f'{column}: {count} invalid ratings found')

            # Replace invalid ratings with null
            dataset.loc[invalid, column] = np.nan
            print(f'{column} is now cleaned with no out of range ratings!')

        else:
            print(f'{column}: no invalid ratings')

In [8]:
# analysing and cleaning Date and Time Formats
def date_checks(dataset, primary_key, date_column_name):

    invalid = []
    valid_all = []

    valid_slash = []
    valid_dot = []
    valid_hyphen = []
    valid_space = []

    ambiguous_slash = []
    ambiguous_dot = []
    ambiguous_hyphen = []
    ambiguous_space = []

    formats = {}

    months = {
        'jan', 'feb', 'mar', 'apr', 'may', 'jun',
        'jul', 'aug', 'sep', 'oct', 'nov', 'dec'
    }

    separator_lists = {
        '/': valid_slash,
        '.': valid_dot,
        '-': valid_hyphen,
        ' ': valid_space
    }

    ambiguous_lists = {
        '/': ambiguous_slash,
        '.': ambiguous_dot,
        '-': ambiguous_hyphen,
        ' ': ambiguous_space
    }

    # ---------------------------------------------------------
    # STEP 1: Structural validation
    # ---------------------------------------------------------

    for primary_id, value in zip(
        dataset[primary_key],
        dataset[date_column_name]
    ):

        if not isinstance(value, str):
            invalid.append(primary_id)
            continue

        separator = next(
            (s for s in ['/', '.', '-', ' '] if s in value),
            None
        )

        if separator is None:
            invalid.append(primary_id)
            continue

        parts = value.split(separator)

        if len(parts) != 3:
            invalid.append(primary_id)
            continue

        first, second, third = parts

        # -----------------------------------------------------
        # DD/MM/YYYY or MM/DD/YYYY
        # -----------------------------------------------------

        if (
            len(first) == 2 and
            len(second) == 2 and
            len(third) == 4
        ):

            try:
                first_num = int(first)
                second_num = int(second)
            except ValueError:
                invalid.append(primary_id)
                continue

            if (
                first_num < 1 or first_num > 31 or
                second_num < 1 or second_num > 31
            ):
                invalid.append(primary_id)

            elif first_num > 12 and second_num > 12:
                invalid.append(primary_id)

            else:
                separator_lists[separator].append(primary_id)

        # -----------------------------------------------------
        # DD/Mon/YYYY
        # -----------------------------------------------------

        elif (
            len(first) == 2 and
            len(second) == 3 and
            len(third) == 4
        ):

            try:
                first_num = int(first)
            except ValueError:
                invalid.append(primary_id)
                continue

            if (
                not 1 <= first_num <= 31 or
                second.lower() not in months
            ):
                invalid.append(primary_id)

            else:
                valid_all.append(primary_id)
                formats[primary_id] = 'DDMonYYYY'

        # -----------------------------------------------------
        # YYYY/MM/DD
        # -----------------------------------------------------

        elif (
            len(first) == 4 and
            len(second) == 2 and
            len(third) == 2
        ):

            try:
                month = int(second)
                day = int(third)
            except ValueError:
                invalid.append(primary_id)
                continue

            if (
                month < 1 or month > 12 or
                day < 1 or day > 31
            ):
                invalid.append(primary_id)

            else:
                valid_all.append(primary_id)
                formats[primary_id] = 'YYYYMMDD'

        else:
            invalid.append(primary_id)

    # ---------------------------------------------------------
    # STEP 2: Whole-group ambiguity check
    # ---------------------------------------------------------

    for separator in ['/', '.', '-', ' ']:

        valid_ids = separator_lists[separator]

        if not valid_ids:
            continue

        data = dataset.loc[
            dataset[primary_key].isin(valid_ids),
            [primary_key, date_column_name]
        ].copy()

        split_dates = data[date_column_name].str.split(separator)

        data['first'] = pd.to_numeric(
            split_dates.str[0],
            errors='coerce'
        )

        data['second'] = pd.to_numeric(
            split_dates.str[1],
            errors='coerce'
        )

        all_first_are_months = data['first'].between(1, 12).all()
        all_second_are_months = data['second'].between(1, 12).all()

        # -----------------------------------------------------
        # Entire group = DD/MM/YYYY
        # -----------------------------------------------------

        if not all_first_are_months and all_second_are_months:

            for primary_id in data[primary_key]:
                formats[primary_id] = 'DDMMYYYY'

        # -----------------------------------------------------
        # Entire group = MM/DD/YYYY
        # -----------------------------------------------------

        elif all_first_are_months and not all_second_are_months:

            for primary_id in data[primary_key]:
                formats[primary_id] = 'MMDDYYYY'

        # -----------------------------------------------------
        # Both columns can contain months
        # -----------------------------------------------------

        elif all_first_are_months and all_second_are_months:

            for _, row in data.iterrows():

                primary_id = row[primary_key]
                first = row['first']
                second = row['second']

                # Same day and month:
                # 05/05/2026, 10/10/2026, etc.
                # Not ambiguous because both interpretations
                # produce the same date.
                if first == second:
                    formats[primary_id] = 'SAME_DAY_MONTH'

                else:
                    # Both can be months but are different.
                    # Therefore DD/MM and MM/DD produce
                    # different dates.
                    ambiguous_lists[separator].append(primary_id)

        # -----------------------------------------------------
        # Mixed group
        # -----------------------------------------------------

        else:

            for _, row in data.iterrows():

                primary_id = row[primary_key]
                first = row['first']
                second = row['second']

                # DD/MM/YYYY
                if first > 12 and 1 <= second <= 12:
                    formats[primary_id] = 'DDMMYYYY'

                # MM/DD/YYYY
                elif second > 12 and 1 <= first <= 12:
                    formats[primary_id] = 'MMDDYYYY'

                # Same day/month
                elif (
                    1 <= first <= 12 and
                    1 <= second <= 12 and
                    first == second
                ):
                    formats[primary_id] = 'SAME_DAY_MONTH'

                # Both can be months but are different
                elif (
                    1 <= first <= 12 and
                    1 <= second <= 12
                ):
                    ambiguous_lists[separator].append(primary_id)

                else:
                    invalid.append(primary_id)

    return {
        'invalid': invalid,
        'valid_all': valid_all,

        'valid_slash': valid_slash,
        'valid_dot': valid_dot,
        'valid_hyphen': valid_hyphen,
        'valid_space': valid_space,

        'ambiguous_slash': ambiguous_slash,
        'ambiguous_dot': ambiguous_dot,
        'ambiguous_hyphen': ambiguous_hyphen,
        'ambiguous_space': ambiguous_space,

        'formats': formats
    }


def clean_dates(dataset, primary_key, date_column_name):

    result = date_checks(
        dataset,
        primary_key,
        date_column_name
    )

    cleaned = dataset.copy()

    # Start everything as NaT
    cleaned[date_column_name] = pd.NaT

    formats = result['formats']

    # ---------------------------------------------------------
    # Parse every ID that was successfully assigned a format
    # ---------------------------------------------------------

    for primary_id, date_format in formats.items():

        mask = cleaned[primary_key] == primary_id

        value = dataset.loc[
            mask,
            date_column_name
        ].iloc[0]

        # Determine separator
        if '/' in value:
            separator = '/'
        elif '.' in value:
            separator = '.'
        elif '-' in value:
            separator = '-'
        elif ' ' in value:
            separator = ' '
        else:
            continue

        # -----------------------------------------------------
        # DD/MM/YYYY
        # -----------------------------------------------------

        if date_format == 'DDMMYYYY':

            parsed = pd.to_datetime(
                value,
                format=f'%d{separator}%m{separator}%Y',
                errors='coerce'
            )

        # -----------------------------------------------------
        # MM/DD/YYYY
        # -----------------------------------------------------

        elif date_format == 'MMDDYYYY':

            parsed = pd.to_datetime(
                value,
                format=f'%m{separator}%d{separator}%Y',
                errors='coerce'
            )

        # -----------------------------------------------------
        # DD/Mon/YYYY
        # -----------------------------------------------------

        elif date_format == 'DDMonYYYY':

            parsed = pd.to_datetime(
                value,
                format=f'%d{separator}%b{separator}%Y',
                errors='coerce'
            )

        # -----------------------------------------------------
        # YYYY/MM/DD
        # -----------------------------------------------------

        elif date_format == 'YYYYMMDD':

            parsed = pd.to_datetime(
                value,
                format=f'%Y{separator}%m{separator}%d',
                errors='coerce'
            )

        # -----------------------------------------------------
        # Same day/month
        # Example: 05/05/2026
        # -----------------------------------------------------

        elif date_format == 'SAME_DAY_MONTH':

            parsed = pd.to_datetime(
                value,
                format=f'%d{separator}%m{separator}%Y',
                errors='coerce'
            )

        else:
            continue

        cleaned.loc[
            mask,
            date_column_name
        ] = parsed

    # ---------------------------------------------------------
    # Ambiguous and invalid IDs remain NaT
    # ---------------------------------------------------------

    ambiguous_ids = (
        result['ambiguous_slash']
        + result['ambiguous_dot']
        + result['ambiguous_hyphen']
        + result['ambiguous_space']
    )

    invalid_ids = result['invalid']

    cleaned.loc[
        cleaned[primary_key].isin(
            ambiguous_ids + invalid_ids
        ),
        date_column_name
    ] = pd.NaT

    return cleaned

In [9]:
# checking valid and invalid time formats
def check_time_formats(dataset, column_name):
    """
    Check whether a column contains valid time values.

    Accepted formats:
        HH:MM
        HH:MM:SS
        H:MM
        H:MM:SS
        HH:MM AM/PM
        HH:MM:SS AM/PM
        H:MM AM/PM
        H:MM:SS AM/PM

    Returns:
        Dictionary containing counts and invalid values.
    """

    valid_count = 0
    invalid_count = 0
    missing_count = 0

    invalid_values = []


    # ---------------------------------------------------------
    # Allowed time formats
    # ---------------------------------------------------------

    time_formats = [
        "%H:%M",
        "%H:%M:%S",
        "%I:%M %p",
        "%I:%M:%S %p"
    ]


    # ---------------------------------------------------------
    # Check every value
    # ---------------------------------------------------------

    for value in dataset[column_name]:

        # Missing values
        if pd.isna(value):
            missing_count += 1
            continue


        value = str(value).strip()


        # Empty strings
        if value == "":
            missing_count += 1
            continue


        # Try all allowed formats
        valid = False

        for fmt in time_formats:

            try:
                datetime.strptime(value, fmt)
                valid = True
                break

            except ValueError:
                continue


        # Record result
        if valid:
            valid_count += 1

        else:
            invalid_count += 1
            invalid_values.append(value)


    # ---------------------------------------------------------
    # Print summary
    # ---------------------------------------------------------

    print(f"Dataset: {column_name}")
    print("-" * 50)

    print(f"Valid times   : {valid_count}")
    print(f"Invalid times : {invalid_count}")
    print(f"Missing times : {missing_count}")


    # ---------------------------------------------------------
    # Display invalid values
    # ---------------------------------------------------------

    if invalid_count > 0:

        print("\nInvalid values:")
        print(pd.Series(invalid_values).value_counts())

    else:

        print("\nNo invalid time values found.")


    # ---------------------------------------------------------
    # Return results
    # ---------------------------------------------------------

    return {
        "valid_count": valid_count,
        "invalid_count": invalid_count,
        "missing_count": missing_count,
        "invalid_values": invalid_values
    }

In [10]:
# cleaning time formats
def clean_time_formats(dataset, column_name):
    """
    Convert valid time values to standardized 24-hour datetime.time values.
    Invalid or missing values are converted to NaT.

    Accepted formats:
        HH:MM
        HH:MM:SS
        HH:MM AM/PM
        HH:MM:SS AM/PM

    Returns:
        The cleaned DataFrame.
    """

    # Keep the original values
    raw_column = f"{column_name}_raw"
    dataset[raw_column] = dataset[column_name].copy()

    time_formats = [
        "%H:%M",
        "%H:%M:%S",
        "%I:%M %p",
        "%I:%M:%S %p"
    ]

    cleaned_times = []

    for value in dataset[column_name]:

        # Missing value
        if pd.isna(value):
            cleaned_times.append(pd.NaT)
            continue

        value = str(value).strip()

        # Empty string
        if value == "":
            cleaned_times.append(pd.NaT)
            continue

        parsed_time = pd.NaT

        # Try each valid format
        for fmt in time_formats:
            try:
                parsed_time = datetime.strptime(value, fmt).time()
                break
            except ValueError:
                continue

        cleaned_times.append(parsed_time)

    # Replace original column with cleaned values
    dataset[column_name] = cleaned_times

    return dataset

In [11]:
# profiling dataset for individual cleaning
def dataset_profiling(dataset):
  results = []
  print('Rows:',dataset.shape[0])
  print('Columns:',dataset.shape[1])
  print('*'*50)
  for column in dataset.columns:
    row = {
        'Column': column,
        'Missing': dataset[column].isna().sum(),
        'Duplicate': dataset[column].duplicated().sum(),
        'Dtype': dataset[column].dtype,
        'unique': dataset[column].nunique()
    }
    results.append(row)
  return pd.DataFrame(results)

In [12]:
# describing all columns in a dataset
def column_description(dataset):
  for col in dataset.columns:
    print('Column:',col)
    print('*'*50)
    print('Description:\n')
    print(dataset[col].describe())

In [13]:
data_path = '/content/drive/MyDrive/Internmo/Zomato1/data/'
datasets = load_datasets(data_path)

In [14]:
basic_dataset_profiling(datasets)

,dataset,rows,columns,duplicates
0,customer_feedback,14200,7,0
1,traffic,18335,6,0
2,weather,18264,7,0
3,order_items,42378,6,0
4,payments,20493,6,0
5,menu,8997,8,0
6,delivery_partners,2000,10,0
7,promotions,300,6,0
8,restaurants,1200,13,0
9,customers,12180,13,180


In [15]:
column_profiling = basic_columns_profiling(datasets)

column_profiling.to_csv('/content/drive/MyDrive/Internmo/Zomato1/column_profile.csv',index=False)

In [16]:
# Creating copies of datasets for cleaning
clean_datasets = {name:df.copy() for name, df in datasets.items()}
print(clean_datasets.keys())

dict_keys(['customer_feedback', 'traffic', 'weather', 'order_items', 'payments', 'menu', 'delivery_partners', 'promotions', 'restaurants', 'customers', 'orders', 'cities'])


In [17]:
# duplicates
# entire row duplicates found
print('count of total rows in customers:', clean_datasets['customers'].shape[0])
print('count of duplicate customers rows:', clean_datasets['customers'].duplicated().sum())

print('count of total rows in orders:', clean_datasets['orders'].shape[0])
print('count of duplicate orders rows:', clean_datasets['orders'].duplicated().sum())

count of total rows in customers: 12180
count of duplicate customers rows: 180
count of total rows in orders: 20705
count of duplicate orders rows: 205


In [18]:
# dropping duplicate rows in customers
before = clean_datasets['customers'].shape[0]
print('rows before removing entire row duplicates in customers', before)
clean_datasets['customers'].drop_duplicates(inplace=True)
after = clean_datasets['customers'].shape[0]
print('rows affected', before - after)
print('rows after removing entire row duplicates in customers', after)

rows before removing entire row duplicates in customers 12180
rows affected 180
rows after removing entire row duplicates in customers 12000


In [19]:
cleaning_decisions(
    dataset = 'customers',
    column = 'all',
    issue = 'exact row duplicates found',
    evidence = 'count of duplicate customers rows: 180',
    action = 'dropped duplicate rows',
    reason = 'no new information is added with duplicate rows',
    before_count = '12180',
    changes_count = '180',
    after_count = '12000',
    status = 'Completed'
)

In [20]:
# dropping duplicate rows in customers
before = clean_datasets['orders'].shape[0]
print('rows before removing entire row duplicates in orders', before)
clean_datasets['orders'].drop_duplicates(inplace=True)
after = clean_datasets['orders'].shape[0]
print('rows affected', before - after)
print('rows after removing entire row duplicates in orders', after)

rows before removing entire row duplicates in orders 20705
rows affected 205
rows after removing entire row duplicates in orders 20500


In [21]:
cleaning_decisions(
    dataset = 'orders',
    column = 'all',
    issue = 'exact row duplicates found',
    evidence = 'count of duplicate orders rows: 205',
    action = 'dropped duplicate rows',
    reason = 'no new information is added with duplicate rows',
    before_count = '20705',
    changes_count = '205',
    after_count = '20500',
    status = 'Completed'
)

In [22]:
# cleaning which spaces around all object columns in all datasets
cleaning_whitespace(clean_datasets)

In [23]:
cleaning_decisions(
    dataset = 'all',
    column = 'all object columns',
    issue = 'white space around object data',
    evidence = 'Example: "Hyderabad" and " Hyderabad " carry the same value but are considered unique',
    action = 'removed white space around all object columns',
    reason = 'cannot have same values considered as unique',
    before_count = 'NA',
    changes_count = 'NA',
    after_count = 'NA',
    status = 'Completed'
)

In [24]:
# foreign key integrity checks
foreign_keys = ['OrderID', 'City', 'FoodItemID', 'RestaurantID', 'CouponCode', 'CustomerID', 'DeliveryPartnerID']
foreign_key_checks = {}

for name, dataset in clean_datasets.items():
    for column in dataset.columns:
        if column in foreign_keys:
            foreign_key_checks.setdefault(name, []).append(column)

foreign_key_checks

{'customer_feedback': ['OrderID'],
 'traffic': ['City'],
 'weather': ['City'],
 'order_items': ['OrderID', 'FoodItemID'],
 'payments': ['OrderID'],
 'menu': ['FoodItemID', 'RestaurantID'],
 'delivery_partners': ['DeliveryPartnerID', 'City'],
 'promotions': ['CouponCode'],
 'restaurants': ['RestaurantID', 'City'],
 'customers': ['CustomerID', 'City'],
 'orders': ['OrderID',
  'CustomerID',
  'RestaurantID',
  'DeliveryPartnerID',
  'CouponCode'],
 'cities': ['City']}

In [25]:
orphan_orders = clean_datasets['customer_feedback'].loc[
    ~clean_datasets['customer_feedback']['OrderID'].isin(clean_datasets['orders']['OrderID']),'OrderID'].unique()
print(orphan_orders.shape[0])

0


In [26]:
orphan_food_item = clean_datasets['order_items'].loc[
    ~clean_datasets['order_items']['FoodItemID'].isin(clean_datasets['menu']['FoodItemID']),'FoodItemID'].unique()
print(orphan_food_item.shape[0])

0


In [27]:
orphan_orders = clean_datasets['order_items'].loc[
    ~clean_datasets['order_items']['OrderID'].isin(clean_datasets['orders']['OrderID']),'OrderID'].unique()
print(orphan_orders.shape[0])

0


In [28]:
orphan_orders = clean_datasets['payments'].loc[
    ~clean_datasets['payments']['OrderID'].isin(clean_datasets['orders']['OrderID']),'OrderID'].unique()
print(orphan_orders.shape[0])

0


In [29]:
orphan_cities = clean_datasets['traffic'].loc[
    ~clean_datasets['traffic']['City'].isin(clean_datasets['cities']['City']),'City'].unique()
print(orphan_cities.shape[0])

0


In [30]:
orphan_cities = clean_datasets['weather'].loc[
    ~clean_datasets['weather']['City'].isin(clean_datasets['cities']['City']),'City'].unique()
print(orphan_cities)

[nan]


In [31]:
orphan_cities = clean_datasets['delivery_partners'].loc[
    ~clean_datasets['delivery_partners']['City'].isin(clean_datasets['cities']['City']),'City'].unique()
print(orphan_cities)

['kolkata' 'ludhiana' 'bhubaneswar' 'DELHI' 'PATNA' 'vadodara' 'NAGPUR'
 'CHENNAI' 'chennai' 'RAIPUR' 'CHANDIGARH' 'SURAT' 'VISAKHAPATNAM'
 'GUWAHATI' 'PUNE' 'JAIPUR' 'New Delhi' 'Bangalore' 'Madras' 'jaipur'
 'KOCHI' 'BANGALORE' 'Bombay' 'BHUBANESWAR' 'BENGALURU' 'VADODARA'
 'KOLKATA' 'guwahati' 'INDORE' 'lucknow' 'delhi' 'hyderabad' 'surat'
 'Baroda' 'coimbatore' 'bengaluru' 'HYDERABAD' 'LUCKNOW' 'MUMBAI'
 'chandigarh' 'bhopal' 'indore' 'nagpur' 'Calcutta' 'raipur' 'AHMEDABAD'
 'LUDHIANA' 'COIMBATORE' 'ahmedabad' 'visakhapatnam' 'patna' 'nashik']


In [32]:
orphan_cities = clean_datasets['restaurants'].loc[
    ~clean_datasets['restaurants']['City'].isin(clean_datasets['cities']['City']),'City'].unique()
print(orphan_cities)


['BHUBANESWAR' 'MUMBAI' 'indore' 'AHMEDABAD' 'bhopal' 'DELHI' 'Madras'
 'nagpur' 'pune' 'SURAT' 'BENGALURU' 'VADODARA' 'INDORE' 'Calcutta'
 'chennai' 'LUDHIANA' 'bengaluru' 'KOLKATA' 'kolkata' 'jaipur'
 'visakhapatnam' 'patna' 'NAGPUR' 'raipur' 'surat' 'nashik' 'BANGALORE'
 'vadodara' 'Bombay' 'HYDERABAD' 'PATNA' 'lucknow' 'Bangalore' 'mumbai'
 'hyderabad' 'delhi' 'CHENNAI' 'guwahati' 'New Delhi' 'kochi' 'NASHIK'
 'KOCHI' 'VISAKHAPATNAM']


In [33]:
orphan_cities = clean_datasets['customers'].loc[
    ~clean_datasets['customers']['City'].isin(clean_datasets['cities']['City']),'City'].unique()
print(orphan_cities)

['DELHI' 'MUMBAI' 'mumbai' 'VADODARA' 'BENGALURU' 'lucknow' 'surat'
 'Madras' 'vadodara' 'VISAKHAPATNAM' 'bhopal' 'delhi' 'NAGPUR'
 'COIMBATORE' 'New Delhi' 'BHOPAL' 'CHANDIGARH' 'guwahati' 'raipur'
 'kolkata' 'CHENNAI' 'kochi' 'KOLKATA' 'bengaluru' 'bhubaneswar' 'INDORE'
 'chandigarh' 'PUNE' 'SURAT' 'LUDHIANA' 'Baroda' 'hyderabad' 'Bangalore'
 'ahmedabad' 'patna' 'Bombay' 'pune' 'jaipur' 'indore' 'PATNA'
 'coimbatore' 'chennai' 'LUCKNOW' 'RAIPUR' 'Calcutta' 'KOCHI' 'AHMEDABAD'
 'nashik' 'BANGALORE' 'visakhapatnam' 'NASHIK' 'BHUBANESWAR' 'ludhiana'
 'nagpur' 'GUWAHATI' 'HYDERABAD' 'JAIPUR']


In [34]:
orphan_restaurants = clean_datasets['menu'].loc[
    ~clean_datasets['menu']['RestaurantID'].isin(clean_datasets['restaurants']['RestaurantID']),'RestaurantID'].unique()
print(orphan_restaurants.shape[0])

0


In [35]:
orphan_restaurants = clean_datasets['orders'].loc[
    ~clean_datasets['orders']['RestaurantID'].isin(clean_datasets['restaurants']['RestaurantID']),'RestaurantID'].unique()
print(orphan_restaurants.shape[0])

13


In [36]:
orphan_delivery_partners = clean_datasets['orders'].loc[
    ~clean_datasets['orders']['DeliveryPartnerID'].isin(clean_datasets['delivery_partners']['DeliveryPartnerID']),'DeliveryPartnerID'].unique()
print(orphan_delivery_partners.shape[0])

0


In [37]:
orphan_coupon_codes = clean_datasets['orders'].loc[
    ~clean_datasets['orders']['CouponCode'].isin(clean_datasets['promotions']['CouponCode']),'CouponCode'].unique()
print(orphan_delivery_partners.shape[0])

0


In [38]:
orphan_restaurants = clean_datasets['orders'].loc[
    ~clean_datasets['orders']['CustomerID'].isin(clean_datasets['customers']['CustomerID']),'CustomerID'].unique()
print(orphan_restaurants.shape[0])

12


In [39]:
# aligning cities in all datasets
orphan_cities = clean_datasets['delivery_partners'].loc[
    ~clean_datasets['delivery_partners']['City'].str.lower().isin(clean_datasets['cities']['City'].str.lower()),'City'].unique()
print(orphan_cities)

['New Delhi' 'Bangalore' 'Madras' 'BANGALORE' 'Bombay' 'Baroda' 'Calcutta']


In [40]:
orphan_cities = clean_datasets['customers'].loc[
    ~clean_datasets['customers']['City'].str.lower().isin(clean_datasets['cities']['City'].str.lower()),'City'].unique()
print(orphan_cities)

['Madras' 'New Delhi' 'Baroda' 'Bangalore' 'Bombay' 'Calcutta' 'BANGALORE']


In [41]:
orphan_cities = clean_datasets['restaurants'].loc[
    ~clean_datasets['restaurants']['City'].str.lower().isin(clean_datasets['cities']['City'].str.lower()),'City'].unique()
print(orphan_cities)

['Madras' 'Calcutta' 'BANGALORE' 'Bombay' 'Bangalore' 'New Delhi']


In [42]:
canonical_cities = clean_datasets['cities']['City'].tolist()
print(canonical_cities)

['Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Chennai', 'Kolkata', 'Pune', 'Ahmedabad', 'Jaipur', 'Lucknow', 'Chandigarh', 'Surat', 'Nagpur', 'Indore', 'Bhopal', 'Coimbatore', 'Kochi', 'Visakhapatnam', 'Patna', 'Bhubaneswar', 'Guwahati', 'Nashik', 'Ludhiana', 'Vadodara', 'Raipur']


In [43]:
city_mapping = {
    'New Delhi':'Delhi',
    'Bangalore':'Bengaluru',
    'Madras':'Chennai',
    'Baroda':'Vadodara',
    'Bombay':'Mumbai',
    'Calcutta':'Kolkata'
}

In [44]:
cleaning_city(clean_datasets,city_mapping)

In [45]:
cleaning_decisions(
    dataset = 'all',
    column = 'City',
    issue = 'not standardised as per master cities dataset with casing issues',
    evidence = 'there are City values not canonically aligned with the City values listed in the cities dataset',
    action = 'standardised the casing to title and mapped the cities with Canonical values',
    reason = 'need consistency across datasets with same values',
    before_count = '>25',
    changes_count = 'NA',
    after_count = '25',
    status = 'Completed'
)

Customer_Feedback Dataset Cleaning Decisions

In [46]:
dataset_profiling(clean_datasets['customer_feedback'])

Rows: 14200
Columns: 7
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,FeedbackID,0,0,int64,14200
1,OrderID,0,115,int64,14085
2,CustomerRating,294,14194,float64,5
3,DeliveryRating,0,14191,int64,9
4,FoodRating,0,14195,int64,5
5,Review,1058,14184,object,15
6,Sentiment,0,14197,object,3


In [47]:
clean_datasets['customer_feedback'].head(10)

,FeedbackID,OrderID,CustomerRating,DeliveryRating,FoodRating,Review,Sentiment
0,1,118680,3.0,2,4,NaN,Neutral
1,2,109542,5.0,5,5,Great food and fast delivery!,Positive
2,3,120380,NaN,3,4,Average experience.,Neutral
3,4,114707,3.0,4,2,NaN,Neutral
4,5,103220,4.0,3,4,Delivery partner was very polite.,Positive
5,6,100679,3.0,3,3,Average experience.,Neutral
6,7,115810,5.0,4,5,Best biryani in town!,Positive
7,8,110236,5.0,4,5,Great food and fast delivery!,Positive
8,9,111565,3.0,2,3,NaN,Neutral
9,10,102005,1.0,2,1,Very disappointed with the service.,Negative


In [48]:
# are there any multiple customer_feedback records for the same order
clean_datasets['customer_feedback']['OrderID'].value_counts().value_counts()

,count
count,
1,13970
2,115


In [49]:
# checking the duplicated OrderID rows
clean_datasets['customer_feedback'][clean_datasets['customer_feedback']['OrderID'].duplicated(keep=False)].sort_values('OrderID')

,FeedbackID,OrderID,CustomerRating,DeliveryRating,FoodRating,Review,Sentiment
2041,2042,100148,5.0,5,4,Amazing experience overall.,Positive
10845,10846,100148,3.0,4,3,"Food was okay, nothing special.",Neutral
12363,12364,100185,NaN,3,2,Order was incomplete.,Negative
931,932,100185,5.0,5,5,"Loved the taste, will order again.",Positive
9498,9499,100508,5.0,4,4,"Loved the taste, will order again.",Positive
...,...,...,...,...,...,...,...
8229,8230,119865,5.0,5,5,Delivery partner was very polite.,Positive
1523,1524,120157,4.0,4,3,Great food and fast delivery!,Positive
11689,11690,120157,1.0,2,1,Food arrived cold and late.,Negative
7513,7514,120437,1.0,1,1,Delivery took way too long.,Negative


In [50]:
# removing duplicate OrderID rows
before = clean_datasets['customer_feedback'].shape[0]
clean_datasets['customer_feedback'] = clean_datasets['customer_feedback'].drop_duplicates(
    subset='OrderID',
    keep=False
)
after = clean_datasets['customer_feedback'].shape[0]

print('count before removing duplicate payment rows:',before)
print('rows affected:', before - after)
print('count after removing duplicate payment rows:',after)

count before removing duplicate payment rows: 14200
rows affected: 230
count after removing duplicate payment rows: 13970


In [51]:
cleaning_decisions(
    dataset = 'customer_feedback',
    column = 'OrderID',
    issue = 'inconsistent duplicate reviews for the same order',
    evidence = '115 duplicated orderid rows and the reviews for the same order are contradicting',
    action = 'removed the entire rows with duplicated rows',
    reason = 'cannot retain contradicting reviews for one order and cannot confirm if the review was edited as there is no timestamp in the dataset.',
    before_count = '14200',
    changes_count = '230',
    after_count = '13970',
    status = 'Completed'
)


In [52]:
# compare if there are any duplicate rows with same information tracked as different rows
compare_columns_without_FeedbackID = []
for col in clean_datasets['customer_feedback'].columns:
  if col != 'FeedbackID':
    compare_columns_without_FeedbackID.append(col)
print(compare_columns_without_FeedbackID)

['OrderID', 'CustomerRating', 'DeliveryRating', 'FoodRating', 'Review', 'Sentiment']


In [53]:
duplicates = clean_datasets['customer_feedback'][
    clean_datasets['customer_feedback'].duplicated(
        subset=compare_columns_without_FeedbackID,
        keep=False
    )
].sort_values('OrderID')

print('count of duplicate rows without FeedbackID:', duplicates.duplicated(subset=compare_columns_without_FeedbackID).sum())

count of duplicate rows without FeedbackID: 0


In [54]:
# foreign key integrity check
clean_datasets['customer_feedback']['OrderID'].isin(clean_datasets['orders']['OrderID']).value_counts()

,count
OrderID,
True,13970


In [55]:
# checking the invalid range for DeliveryRating column
clean_datasets['customer_feedback'][clean_datasets['customer_feedback']['DeliveryRating']<1].shape[0]

0

In [56]:
clean_datasets['customer_feedback'][clean_datasets['customer_feedback']['DeliveryRating']>5].shape[0]

126

In [57]:
# checking the invalid range for DeliveryRating column
clean_datasets['customer_feedback'][clean_datasets['customer_feedback']['CustomerRating']<1].shape[0]

0

In [58]:
clean_datasets['customer_feedback'][clean_datasets['customer_feedback']['CustomerRating']>5].shape[0]

0

In [59]:
# checking the invalid range for DeliveryRating column
clean_datasets['customer_feedback'][clean_datasets['customer_feedback']['FoodRating']<1].shape[0]

0

In [60]:
clean_datasets['customer_feedback'][clean_datasets['customer_feedback']['FoodRating']>5].shape[0]

0

In [61]:
# cleaning all the rating columns in the customer_feedback dataset
clean_rating(clean_datasets['customer_feedback'],['DeliveryRating','CustomerRating','FoodRating'])

DeliveryRating: 126 invalid ratings found
DeliveryRating is now cleaned with no out of range ratings!
CustomerRating: no invalid ratings
FoodRating: no invalid ratings


In [62]:
cleaning_decisions(
    dataset = 'customer_feedback',
    column = 'DeliveryRating',
    issue = 'ratings >5 found',
    evidence = 'DeliveryRating: 126 invalid ratings found',
    action = 'converted the invalid range values to NaN',
    reason = 'cannot retain invalid data',
    before_count = '126',
    changes_count = '126',
    after_count = '0',
    status = 'Completed'
)

In [63]:
clean_datasets['customer_feedback']['DeliveryRating'].value_counts()

,count
DeliveryRating,
5.0,4179
4.0,3452
3.0,2583
2.0,2086
1.0,1544


In [64]:
clean_datasets['customer_feedback']['CustomerRating'].value_counts()

,count
CustomerRating,
5.0,4972
3.0,2667
4.0,2570
2.0,2345
1.0,1129


In [65]:
clean_datasets['customer_feedback']['FoodRating'].value_counts()

,count
FoodRating,
5,4199
4,3555
3,2582
2,2091
1,1543


In [66]:
clean_datasets['customer_feedback']['Review'].value_counts()

,count
Review,
Amazing experience overall.,1275
Packaging was excellent and food was hot.,1272
Best biryani in town!,1264
Great food and fast delivery!,1235
"Loved the taste, will order again.",1229
Delivery partner was very polite.,1221
"Food was okay, nothing special.",677
Delivery was on time but food was mediocre.,675
Average experience.,649


In [67]:
clean_datasets['customer_feedback']['Sentiment'].value_counts()

,count
Sentiment,
Positive,7716
Negative,3540
Neutral,2714


In [68]:
dataset_profiling(clean_datasets['customer_feedback'])

Rows: 13970
Columns: 7
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,FeedbackID,0,0,int64,13970
1,OrderID,0,0,int64,13970
2,CustomerRating,287,13964,float64,5
3,DeliveryRating,126,13964,float64,5
4,FoodRating,0,13965,int64,5
5,Review,1045,13954,object,15
6,Sentiment,0,13967,object,3


In [69]:
cleaning_decisions(
    dataset = 'customer_feedback',
    column = 'CustomerRating',
    issue = 'missing',
    evidence = '287 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '287',
    changes_count = '0',
    after_count = '287',
    status = 'Profiled'
)

In [70]:
cleaning_decisions(
    dataset = 'customer_feedback',
    column = 'DeliveryRating',
    issue = 'missing',
    evidence = '126 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '126',
    changes_count = '0',
    after_count = '126',
    status = 'Profiled'
)

In [71]:
cleaning_decisions(
    dataset = 'customer_feedback',
    column = 'Review',
    issue = 'missing',
    evidence = '1045 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '1045',
    changes_count = '0',
    after_count = '1045',
    status = 'Profiled'
)

In [72]:
column_description(clean_datasets['customer_feedback'])

Column: FeedbackID
**************************************************
Description:

count    13970.000000
mean      7099.488261
std       4099.608340
min          1.000000
25%       3543.250000
50%       7100.500000
75%      10650.750000
max      14200.000000
Name: FeedbackID, dtype: float64
Column: OrderID
**************************************************
Description:

count     13970.000000
mean     110262.770079
std        5908.956095
min      100001.000000
25%      105143.500000
50%      110276.500000
75%      115375.500000
max      120500.000000
Name: OrderID, dtype: float64
Column: CustomerRating
**************************************************
Description:

count    13683.000000
mean         3.578163
std          1.344839
min          1.000000
25%          2.000000
50%          4.000000
75%          5.000000
max          5.000000
Name: CustomerRating, dtype: float64
Column: DeliveryRating
**************************************************
Description:

count    13844.000000
m

traffic dataset cleaning decisions

In [73]:
dataset_profiling(clean_datasets['traffic'])

Rows: 18335
Columns: 6
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,TrafficID,0,0,int64,18335
1,City,0,18310,object,25
2,Date,0,14976,object,3359
3,Time,0,18239,object,96
4,TrafficLevel,147,18330,object,4
5,AverageSpeed,0,17566,float64,769


In [74]:
# checking if the City values present in the traffic dataset align with the City values present in the cities dataset
clean_datasets['traffic']['City'].isin(clean_datasets['cities']['City']).value_counts()

,count
City,
True,18335


In [75]:
# analysing the available formats in the Date column in the traffic dataset
traffic_dates = date_checks(clean_datasets['traffic'],'TrafficID','Date')
for validity, ids in traffic_dates.items():
  print(f'{validity} count: {len(ids)}')

invalid count: 0
valid_all count: 7431
valid_slash count: 7240
valid_dot count: 0
valid_hyphen count: 3664
valid_space count: 0
ambiguous_slash count: 2557
ambiguous_dot count: 0
ambiguous_hyphen count: 0
ambiguous_space count: 0
formats count: 15778


In [76]:
# parsing the unambiguous Date values
clean_datasets['traffic'] = clean_dates(clean_datasets['traffic'],'TrafficID','Date')

In [77]:
cleaning_decisions(
    dataset = 'traffic',
    column = 'Date',
    issue = 'multiple date formats mixed DD/MM and MM/DD',
    evidence = '2557 date values are ambiguous where we cannot determine if the date format is DD/MM or MM/DD',
    action = 'converted the ambiguous Date formats to NaT',
    reason = 'cannot assume the Date values with multiple formats across the column',
    before_count = '2557',
    changes_count = '2557',
    after_count = '0',
    status = 'Completed'
)

In [78]:
# checking if there are any invalid time formats
check_time_formats(clean_datasets['traffic'],'Time')

Dataset: Time
--------------------------------------------------
Valid times   : 18335
Invalid times : 0
Missing times : 0

No invalid time values found.


{'valid_count': 18335,
 'invalid_count': 0,
 'missing_count': 0,
 'invalid_values': []}

In [79]:
# checking missing TrafficLevel values in the traffic dataset
clean_datasets['traffic'][clean_datasets['traffic']['TrafficLevel'].isna()]

,TrafficID,City,Date,Time,TrafficLevel,AverageSpeed
108,109,Mumbai,2024-08-29,06:30,NaN,20.8
137,138,Mumbai,2024-06-20,18:45,NaN,16.5
279,280,Mumbai,2024-06-10,08:30,NaN,15.6
321,322,Mumbai,2023-02-21,19:45,NaN,24.6
334,335,Mumbai,2023-08-23,08:30,NaN,4.8
...,...,...,...,...,...,...
17787,17788,Raipur,2024-09-14,18:15,NaN,21.3
17857,17858,Raipur,NaT,21:30,NaN,30.3
17880,17881,Raipur,2023-08-31,08:30,NaN,12.1
18115,18116,Raipur,2024-07-08,14:15,NaN,15.7


In [80]:
cleaning_decisions(
    dataset = 'traffic',
    column = 'TrafficLevel',
    issue = 'missing',
    evidence = '147 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '147',
    changes_count = '0',
    after_count = '147',
    status = 'Profiled'
)

In [81]:
# checking invalid range of AverageSpeed values
(clean_datasets['traffic']['AverageSpeed']<0).value_counts()

,count
AverageSpeed,
False,18149
True,186


In [82]:
# removing negative values in the AverageSpeed column
clean_datasets['traffic'].loc[clean_datasets['traffic']['AverageSpeed']<0,'AverageSpeed'] = np.nan

In [83]:
cleaning_decisions(
    dataset = 'traffic',
    column = 'AverageSpeed',
    issue = 'negative AverageSpeed values',
    evidence = '186 negative AverageSpeed values in the traffic dataset',
    action = 'converted the negative AverageSpeed values to NaN',
    reason = 'cannot retain invalid values and there is no way to confirm the actual value',
    before_count = '186',
    changes_count = '186',
    after_count = '0',
    status = 'Completed'
)

In [84]:
column_description(clean_datasets['traffic'])

Column: TrafficID
**************************************************
Description:

count    18335.000000
mean      9168.000000
std       5293.002928
min          1.000000
25%       4584.500000
50%       9168.000000
75%      13751.500000
max      18335.000000
Name: TrafficID, dtype: float64
Column: City
**************************************************
Description:

count      18335
unique        25
top       Mumbai
freq         759
Name: City, dtype: object
Column: Date
**************************************************
Description:

count                            15778
mean     2024-01-04 12:35:52.059830272
min                2023-01-01 00:00:00
25%                2023-07-08 00:00:00
50%                2024-01-06 00:00:00
75%                2024-07-05 00:00:00
max                2024-12-31 00:00:00
Name: Date, dtype: object
Column: Time
**************************************************
Description:

count     18335
unique       96
top       08:45
freq        453
Name: Time, dtype:

Weather Dataset Cleaning Decisions

In [85]:
dataset_profiling(clean_datasets['weather'])

Rows: 18264
Columns: 7
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,WeatherID,0,0,int64,18264
1,City,91,18238,object,25
2,Date,0,14915,object,3349
3,Temperature,0,17798,float64,466
4,Rainfall,269,15914,float64,2349
5,Humidity,0,18115,int64,149
6,WeatherCondition,0,18258,object,6


In [86]:
# checking if the City values in the weather dataset align with the City values in the cities dataset
clean_datasets['weather']['City'].notna().isin(clean_datasets['cities']['City']).value_counts()

,count
City,
False,18264


In [87]:
clean_datasets['weather'][clean_datasets['weather']['City'].isna()]

,WeatherID,City,Date,Temperature,Rainfall,Humidity,WeatherCondition
260,261,NaN,2024-03-18,29.6,9.2,50,Rainy
354,355,NaN,10/07/2023,26.8,0.0,56,Cloudy
640,641,NaN,01/20/2023,15.6,0.0,64,Rainy
702,703,NaN,01/31/2024,17.6,6.2,72,Foggy
801,802,NaN,2024-03-12,36.3,3.2,82,Rainy
...,...,...,...,...,...,...,...
17494,17495,NaN,12/14/2024,13.6,7.8,61,Foggy
17499,17500,NaN,22/08/2024,36.2,0.0,69,Stormy
17815,17816,NaN,06-12-2023,21.3,0.9,36,Foggy
18039,18040,NaN,12/09/2024,25.1,25.0,66,Stormy


In [88]:
# removing rows where the City values are NaN
clean_datasets['weather'] = clean_datasets['weather'].dropna(subset=['City'])

In [89]:
cleaning_decisions(
    dataset = 'weather',
    column = 'City',
    issue = 'missing City values',
    evidence = 'there are 91 rows with missing City values',
    action = 'removed the entire rows with missing City values',
    reason = 'weather dataset without City values do not give enough information on the weather of the city',
    before_count = '18264',
    changes_count = '91',
    after_count = '18173',
    status = 'Completed'
)

In [90]:
# analysing date formats in the Date column of the weather dataset
weather_dates = date_checks(clean_datasets['weather'],'WeatherID','Date')
for validity, ids in weather_dates.items():
  print(f'{validity} count: {len(ids)}')


invalid count: 0
valid_all count: 7220
valid_slash count: 7297
valid_dot count: 0
valid_hyphen count: 3656
valid_space count: 0
ambiguous_slash count: 2582
ambiguous_dot count: 0
ambiguous_hyphen count: 0
ambiguous_space count: 0
formats count: 15591


In [91]:
# parsing the unambiguous date formats in the Date column
clean_datasets['weather'] = clean_dates(clean_datasets['weather'],'WeatherID','Date')

In [92]:
cleaning_decisions(
    dataset = 'weather',
    column = 'Date',
    issue = 'multiple date formats mixed DD/MM and MM/DD',
    evidence = '2582 date values are ambiguous where we cannot determine if the date format is DD/MM or MM/DD',
    action = 'converted the ambiguous Date formats to NaT',
    reason = 'cannot assume the Date values with multiple formats across the column',
    before_count = '2582',
    changes_count = '2582',
    after_count = '0',
    status = 'Completed'
)

In [93]:
# describing all the columns in the weather dataset
column_description(clean_datasets['weather'])

Column: WeatherID
**************************************************
Description:

count    18173.000000
mean      9131.134210
std       5271.607481
min          1.000000
25%       4565.000000
50%       9130.000000
75%      13698.000000
max      18264.000000
Name: WeatherID, dtype: float64
Column: City
**************************************************
Description:

count      18173
unique        25
top       Indore
freq         754
Name: City, dtype: object
Column: Date
**************************************************
Description:

count                            15591
mean     2024-01-05 06:08:31.218010624
min                2023-01-01 00:00:00
25%                2023-07-10 00:00:00
50%                2024-01-06 00:00:00
75%                2024-07-03 00:00:00
max                2024-12-31 00:00:00
Name: Date, dtype: object
Column: Temperature
**************************************************
Description:

count    18173.000000
mean        26.801695
std          7.298632
min      

In [94]:
# checking negative Humidity values
clean_datasets['weather'][clean_datasets['weather']['Humidity']<0].head(20)

,WeatherID,City,Date,Temperature,Rainfall,Humidity,WeatherCondition
173,174,Mumbai,NaT,16.0,7.4,-76,Clear
258,259,Mumbai,2024-03-03,39.2,7.6,-61,Cloudy
269,270,Mumbai,2024-09-28,32.4,78.8,-48,Clear
514,515,Mumbai,2023-11-28,34.0,0.0,-59,Cloudy
569,570,Mumbai,2024-05-30,34.8,9.0,-63,Cloudy
631,632,Mumbai,2023-03-15,30.5,2.1,-67,Cloudy
690,691,Mumbai,2024-01-16,16.2,4.5,-63,Cloudy
817,818,Delhi,2024-10-10,28.4,14.5,-64,Clear
837,838,Delhi,2024-02-20,12.5,12.1,-54,Foggy
865,866,Delhi,2024-08-22,31.2,15.0,-69,Stormy


In [95]:
(clean_datasets['weather']['Humidity']<0).value_counts()

,count
Humidity,
False,17984
True,189


In [96]:
clean_datasets['weather'][clean_datasets['weather']['Humidity']<0].describe()

,WeatherID,Date,Temperature,Rainfall,Humidity
count,189.000000,160,189.000000,187.000000,189.000000
mean,8845.687831,2023-12-20 23:15:00,27.204762,26.190374,-59.116402
min,174.000000,2023-01-01 00:00:00,9.700000,0.000000,-97.000000
25%,4103.000000,2023-06-15 18:00:00,22.800000,0.000000,-69.000000
50%,8773.000000,2023-12-02 00:00:00,27.900000,5.100000,-60.000000
75%,13766.000000,2024-06-15 12:00:00,32.100000,20.250000,-49.000000
max,18251.000000,2024-12-24 00:00:00,42.300000,365.300000,-10.000000
std,5365.492604,NaN,7.127966,54.271160,14.433162


In [97]:
# converting negative Humidity values to NaN
invalid = clean_datasets['weather']['Humidity'] < 0
clean_datasets['weather'].loc[invalid, 'Humidity'] = np.nan

In [98]:
cleaning_decisions(
    dataset = 'weather',
    column = 'Humidity',
    issue = 'negative Humidity values',
    evidence = '189 negative Humidity values',
    action = 'converted the negative values to NaN',
    reason = 'cannot retain invalid range values',
    before_count = '189',
    changes_count = '189',
    after_count = '0',
    status = 'Completed'
)

In [99]:
clean_datasets['weather']['WeatherCondition'].value_counts()

,count
WeatherCondition,
Clear,6332
Cloudy,3650
Humid,2711
Rainy,2702
Foggy,1864
Stormy,914


In [100]:
clean_datasets['weather']['Rainfall'].isna().value_counts()

,count
Rainfall,
False,17905
True,268


In [101]:
cleaning_decisions(
    dataset = 'weather',
    column = 'Rainfall',
    issue = 'missing',
    evidence = '268 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '268',
    changes_count = '0',
    after_count = '268',
    status = 'Profiled'
)

order_items dataset cleaning decisions

In [102]:
dataset_profiling(clean_datasets['order_items'])

Rows: 42378
Columns: 6
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,OrderItemID,0,0,int64,42378
1,OrderID,0,21891,int64,20487
2,FoodItemID,0,33546,int64,8832
3,Quantity,0,42370,int64,8
4,UnitPrice,0,41980,int64,398
5,TotalPrice,435,41429,float64,948


In [103]:
# checking foreign key integrity
clean_datasets['order_items']['OrderID'].isin(clean_datasets['orders']['OrderID']).value_counts()

,count
OrderID,
True,42378


In [104]:
clean_datasets['order_items']['FoodItemID'].isin(clean_datasets['menu']['FoodItemID']).value_counts()

,count
FoodItemID,
True,42378


In [105]:
# checking in-valid Quantity values in the order_items dataset
negative_quantity = clean_datasets['order_items'][clean_datasets['order_items']['Quantity']<0]
positive_quantity = clean_datasets['order_items'][clean_datasets['order_items']['Quantity']>0]

In [106]:
negative_quantity.shape[0]

417

In [107]:
(positive_quantity['Quantity']*positive_quantity['UnitPrice'] == positive_quantity['TotalPrice']).value_counts()

,count
True,41532
False,429


In [108]:
negative_quantity[negative_quantity['Quantity'].abs()*negative_quantity['UnitPrice'] != negative_quantity['TotalPrice']].sort_values('TotalPrice')

,OrderItemID,OrderID,FoodItemID,Quantity,UnitPrice,TotalPrice
1211,1212,100590,2245,-4,123,NaN
2003,2004,100982,2360,-1,207,NaN
8871,8872,104359,257,-1,136,NaN
15775,15776,107701,7416,-2,153,NaN
38575,38576,118819,495,-1,109,NaN
39171,39172,119112,8349,-1,305,NaN


In [109]:
positive_quantity[positive_quantity['Quantity']*positive_quantity['UnitPrice'] != positive_quantity['TotalPrice']].sort_values('TotalPrice')

,OrderItemID,OrderID,FoodItemID,Quantity,UnitPrice,TotalPrice
74,75,100040,8957,3,173,NaN
197,198,100099,7003,1,305,NaN
297,298,100143,5217,2,139,NaN
519,520,100259,3936,2,198,NaN
544,545,100273,1438,4,76,NaN
...,...,...,...,...,...,...
41893,41894,120454,3255,1,221,NaN
42311,42312,101662,5823,1,156,NaN
42327,42328,115660,1131,1,208,NaN
42364,42365,110265,3033,1,89,NaN


In [110]:
# converting the quantity values to absolute
clean_datasets['order_items'].loc[clean_datasets['order_items']['Quantity']<0,'Quantity'] = clean_datasets['order_items']['Quantity'].abs()

In [111]:
cleaning_decisions(
    dataset = 'order_items',
    column = 'Quantity',
    issue = 'negative values',
    evidence = 'there are 417 negative values in the Quantity column of order_items dataset',
    action = 'converted to the absolute values',
    reason = 'with enough evidence that the total price is quantity times unitprice for all the available total price values',
    before_count = '417',
    changes_count = '417',
    after_count = '0',
    status = 'Completed'
)


In [112]:
# converting null values of the TotalPrice to Quantity * UnitPrice
clean_datasets['order_items'].loc[clean_datasets['order_items']['TotalPrice'].isna(),'TotalPrice'] = clean_datasets['order_items']['Quantity'] * clean_datasets['order_items']['UnitPrice']

In [113]:
cleaning_decisions(
    dataset = 'order_items',
    column = 'TotalPrice',
    issue = 'missing TotalPrice values',
    evidence = 'there are 435 missing values in the TotalPrice column',
    action = 'replaced with the calculation based on TotalPrice = Quantity * UnitPrice',
    reason = 'enough evidence to calculate missing TotalPrice values',
    before_count = '435',
    changes_count = '435',
    after_count = '0',
    status = 'Completed'
)

In [114]:
# comparing UnitPrice values with the listed Price in menu dataset
clean_datasets['menu'].head(5)

,FoodItemID,RestaurantID,FoodName,Category,Price,PreparationTime,Calories,Availability
0,1,1,Biryani,Main Course,220,16,519.0,Yes
1,2,1,Cheese Pizza,Fast Food,123,13,215.0,Yes
2,3,1,Veg Sandwich,Fast Food,40,28,440.0,Yes
3,4,1,Butter Naan,Breads,227,16,683.0,No
4,5,1,Garlic Naan,Breads,338,17,395.0,Yes


In [115]:
price_check = clean_datasets['order_items'].merge(
    clean_datasets['menu'][['FoodItemID','Price']],
    on='FoodItemID',
    how='left'
)

In [116]:
price_check.head()

,OrderItemID,OrderID,FoodItemID,Quantity,UnitPrice,TotalPrice,Price
0,1,100001,3539,3,194,582.0,194
1,2,100001,3539,1,194,194.0,194
2,3,100002,6183,3,238,714.0,238
3,4,100002,6182,1,371,371.0,371
4,5,100003,771,1,114,114.0,114


In [117]:
(price_check['UnitPrice'] == price_check['Price']).value_counts()

,count
True,41942
False,436


In [118]:
# checking the Price values not matching with the UnitPrice values
price_check[price_check['UnitPrice'] != price_check['Price']]

,OrderItemID,OrderID,FoodItemID,Quantity,UnitPrice,TotalPrice,Price
127,128,100064,3404,4,218,872.0,-218
352,353,100172,2386,1,358,358.0,-358
428,429,100208,6159,1,193,193.0,-193
438,439,100215,2568,3,233,699.0,-233
439,440,100215,2568,1,233,233.0,-233
...,...,...,...,...,...,...,...
41687,41688,120353,3467,1,135,135.0,-135
41798,41799,120409,2099,1,289,289.0,-289
41820,41821,120418,5897,3,210,630.0,-210
42074,42075,117569,8133,2,342,684.0,-342


In [119]:
# are there any more non-matching values than negative values
price_check[price_check['UnitPrice'] != price_check['Price'].abs()]

,OrderItemID,OrderID,FoodItemID,Quantity,UnitPrice,TotalPrice,Price


In [120]:
column_description(clean_datasets['order_items'])

Column: OrderItemID
**************************************************
Description:

count    42378.000000
mean     21189.500000
std      12233.619191
min          1.000000
25%      10595.250000
50%      21189.500000
75%      31783.750000
max      42378.000000
Name: OrderItemID, dtype: float64
Column: OrderID
**************************************************
Description:

count     42378.000000
mean     110250.926188
std        5913.093682
min      100001.000000
25%      105122.250000
50%      110265.000000
75%      115380.000000
max      120500.000000
Name: OrderID, dtype: float64
Column: FoodItemID
**************************************************
Description:

count    42378.000000
mean      4498.668224
std       2605.663285
min          1.000000
25%       2236.000000
50%       4483.500000
75%       6754.000000
max       8997.000000
Name: FoodItemID, dtype: float64
Column: Quantity
**************************************************
Description:

count    42378.000000
mean         

Payments Dataset Profiling

In [121]:
dataset_profiling(clean_datasets['payments'])

Rows: 20493
Columns: 6
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,PaymentID,0,0,int64,20493
1,OrderID,0,399,int64,20094
2,PaymentMethod,0,20487,object,6
3,PaymentStatus,0,20489,object,4
4,TransactionID,183,383,object,20109
5,PaymentDate,0,17136,object,3357


In [122]:
# Hypothesis - there requires to be one successful payment per order
# payments dataset TransactionID duplicate checks
clean_datasets['payments'][clean_datasets['payments']['TransactionID'].duplicated(keep=False)].sort_values('TransactionID')

,PaymentID,OrderID,PaymentMethod,PaymentStatus,TransactionID,PaymentDate
20362,20363,117399,Wallet,Success,TXN01836679JX,02/15/2023
6720,6721,117399,Wallet,Success,TXN01836679JX,02/15/2023
20333,20334,102990,Cash on Delivery,Success,TXN03057903AN,2023.03.18
11955,11956,102990,Cash on Delivery,Success,TXN03057903AN,2023.03.18
4682,4683,116289,Credit Card,Success,TXN03854803LE,2023-07-21
...,...,...,...,...,...,...
20203,20204,101277,Debit Card,Success,NaN,01/04/2024
20213,20214,120211,Wallet,Success,NaN,08-09-2023
20229,20230,119709,Wallet,Success,NaN,2023-01-21
20278,20279,101919,Credit Card,Success,NaN,2023-09-26


In [123]:
# checking if there are any duplicate rows of data without considering the Primary Key PaymentID
compare_columns_without_PaymentID = [
    col for col in clean_datasets['payments'].columns
    if col != 'PaymentID'
]
print(compare_columns_without_PaymentID)

['OrderID', 'PaymentMethod', 'PaymentStatus', 'TransactionID', 'PaymentDate']


In [124]:
duplicates = clean_datasets['payments'][
    clean_datasets['payments'].duplicated(
        subset=compare_columns_without_PaymentID,
        keep=False
    )
].sort_values('TransactionID')

print('count of duplicate rows without PaymentID:', duplicates.duplicated(subset=compare_columns_without_PaymentID).sum())

count of duplicate rows without PaymentID: 202


In [125]:
# dropping duplicates where all the columns except payment id is same

before = clean_datasets['payments'].shape[0]
clean_datasets['payments'] = clean_datasets['payments'].drop_duplicates(
    subset=compare_columns_without_PaymentID,
    keep='first'
)
after = clean_datasets['payments'].shape[0]

print('count before removing duplicate payment rows:',before)
print('rows affected:', before - after)
print('count after removing duplicate payment rows:',after)

count before removing duplicate payment rows: 20493
rows affected: 202
count after removing duplicate payment rows: 20291


In [126]:
cleaning_decisions(
    dataset = 'payments',
    column = 'all',
    issue = 'same information tracked under different PaymentID values',
    evidence = 'there are 202 duplicate rows of data giving the same information on an order',
    action = 'dropping entire duplicate rows and retaining the 1st occurance',
    reason = 'retaining duplicate information adds no new information and hence no value',
    before_count = '202',
    changes_count = '202',
    after_count = '0',
    status = 'Completed'
)

In [127]:
# payments dataset duplicate re-checks
for column in clean_datasets['payments'].columns:
  print(f'{column}: ',clean_datasets['payments'][column].duplicated(keep=False).sum())

PaymentID:  0
OrderID:  394
PaymentMethod:  20291
PaymentStatus:  20291
TransactionID:  182
PaymentDate:  20240


In [128]:
# how many duplicate TransactionID values exist that are also null
print('count of null/duplicates in TransactionID:', (clean_datasets['payments']['TransactionID'].duplicated(keep=False) & clean_datasets['payments']['TransactionID'].isna()).sum())

count of null/duplicates in TransactionID: 182


In [129]:
# checking duplicated OrderID values with the PaymentStatus not as Success
not_success = clean_datasets['payments'][(clean_datasets['payments'].duplicated(subset=['OrderID'],keep=False))
& (clean_datasets['payments']['PaymentStatus'] != 'Success')].sort_values('OrderID')

In [130]:
not_success.shape[0]

78

In [131]:
not_success[not_success['OrderID'].duplicated(keep=False)]

,PaymentID,OrderID,PaymentMethod,PaymentStatus,TransactionID,PaymentDate
6009,6010,104650,UPI,Refunded,TXN19062440TS,2023.03.26
20194,20195,104650,UPI,Pending,TXN01395402JT,2023.03.26
9150,9151,105567,UPI,Refunded,TXN34531171WF,09/14/2024
15539,15540,105567,UPI,Failed,TXN42903360KT,09/14/2024
17952,17953,116334,Credit Card,Refunded,TXN55221266OL,26-05-2023
10203,10204,116334,Credit Card,Refunded,TXN01670958NM,2023.05.26


In [132]:
# checking duplicated OrderID values with the PaymentStatus as Success
success = clean_datasets['payments'][(clean_datasets['payments'].duplicated(subset=['OrderID'],keep=False))
& (clean_datasets['payments']['PaymentStatus'] == 'Success')].sort_values('OrderID')

In [133]:
success[success['OrderID'].duplicated(keep=False)]

,PaymentID,OrderID,PaymentMethod,PaymentStatus,TransactionID,PaymentDate
8891,8892,100146,Credit Card,Success,TXN20234022PI,2023.12.26
4767,4768,100146,Credit Card,Success,TXN86658265JB,26/12/2023
20145,20146,100148,Wallet,Success,TXN07345168UQ,28/07/2024
15155,15156,100148,Wallet,Success,TXN95497450IP,2024.07.28
3260,3261,100185,UPI,Success,TXN89943353SN,2023.05.01
...,...,...,...,...,...,...
3144,3145,119865,Debit Card,Success,TXN13172886WF,26-05-2024
86,87,120278,Cash on Delivery,Success,TXN76706014GZ,07/16/2024
13756,13757,120278,Cash on Delivery,Success,TXN73915414FW,2024-07-16
7767,7768,120437,Credit Card,Success,TXN23149050JK,17/12/2024


In [134]:
success.shape[0]

316

In [135]:
cleaning_decisions(
    dataset = 'payments',
    column = 'OrderID',
    issue = 'duplicate successful order payments found with different TransactionID values',
    evidence = 'there are 246 duplicate OrderID with same PaymentStatus 148 genuine duplicate OrderID values',
    action = 'retained all the duplicate OrderID entries',
    reason = 'cannot confirm the correct TransactionID for the duplicate OrderID',
    before_count = '394',
    changes_count = '0',
    after_count = '394',
    status = 'Profiled'
)

In [136]:
cleaning_decisions(
    dataset = 'payments',
    column = 'TransactionID',
    issue = 'Null TransactionID values',
    evidence = 'there are 182 Null values that are the only duplicates',
    action = 'Retained',
    reason = 'cannot fabricate TransactionID values',
    before_count = '182',
    changes_count = '0',
    after_count = '182',
    status = 'Profiled'
)

In [137]:
# categories present in PaymentStatus column
clean_datasets['payments']['PaymentStatus'].value_counts()

,count
PaymentStatus,
Success,16209
Failed,1645
Refunded,1627
Pending,810


In [138]:
# analysing date formats present in PaymentDate column
payment_dates = date_checks(clean_datasets['payments'],'PaymentID','PaymentDate')
for validity, ids in traffic_dates.items():
  print(f'{validity} count: {len(ids)}')


invalid count: 0
valid_all count: 7431
valid_slash count: 7240
valid_dot count: 0
valid_hyphen count: 3664
valid_space count: 0
ambiguous_slash count: 2557
ambiguous_dot count: 0
ambiguous_hyphen count: 0
ambiguous_space count: 0
formats count: 15778


In [139]:
# Parsing the valid and unambiguous PaymentDate values in the payments dataset
clean_datasets['payments'] = clean_dates(clean_datasets['payments'],'PaymentID','PaymentDate')

In [140]:
cleaning_decisions(
    dataset = 'payments',
    column = 'PaymentDate',
    issue = 'multiple date formats mixed DD/MM and MM/DD',
    evidence = '2557 date values are ambiguous where we cannot determine if the date format is DD/MM or MM/DD',
    action = 'converted the ambiguous Date formats to NaT',
    reason = 'cannot assume the Date values with multiple formats across the column',
    before_count = '2557',
    changes_count = '2557',
    after_count = '0',
    status = 'Completed'
)

In [141]:
# Foreign key integrity checks
clean_datasets['payments']['OrderID'].isin(clean_datasets['orders']['OrderID']).value_counts()

,count
OrderID,
True,20291


In [142]:
# describing payments dataset
column_description(clean_datasets['payments'])

Column: PaymentID
**************************************************
Description:

count    20291.000000
mean     10146.000000
std       5857.651492
min          1.000000
25%       5073.500000
50%      10146.000000
75%      15218.500000
max      20291.000000
Name: PaymentID, dtype: float64
Column: OrderID
**************************************************
Description:

count     20291.000000
mean     110257.776600
std        5917.629149
min      100001.000000
25%      105130.500000
50%      110265.000000
75%      115388.500000
max      120500.000000
Name: OrderID, dtype: float64
Column: PaymentMethod
**************************************************
Description:

count          20291
unique             6
top       Debit Card
freq            3501
Name: PaymentMethod, dtype: object
Column: PaymentStatus
**************************************************
Description:

count       20291
unique          4
top       Success
freq        16209
Name: PaymentStatus, dtype: object
Column: Transa

menu dataset cleaning decisions

In [143]:
dataset_profiling(clean_datasets['menu'])

Rows: 8997
Columns: 8
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,FoodItemID,0,0,int64,8997
1,RestaurantID,0,7797,int64,1200
2,FoodName,0,8904,object,93
3,Category,0,8991,object,6
4,Price,0,8532,int64,465
5,PreparationTime,0,8951,int64,46
6,Calories,132,8237,float64,759
7,Availability,0,8995,object,2


In [144]:
# ForeignKey integrity checks
clean_datasets['menu']['RestaurantID'].isin(clean_datasets['restaurants']['RestaurantID']).value_counts()

,count
RestaurantID,
True,8997


In [145]:
clean_datasets['menu']['FoodName'].str.strip().str.title().nunique()

31

In [146]:
# standardising the casing of FoodName
clean_datasets['menu']['FoodName'] = clean_datasets['menu']['FoodName'].str.strip().str.title()

In [147]:
cleaning_decisions(
    dataset = 'menu',
    column = 'FoodName',
    issue = 'inconsistent casing found',
    evidence = 'there are 93 unique values with inconsisted casing in FoodName column',
    action = 'standardised casing to title across the column',
    reason = 'cannot have duplicate entries considered as unique because of casing inconsistencies',
    before_count = '93',
    changes_count = '62',
    after_count = '31',
    status = 'Completed'
)

In [148]:
clean_datasets['menu']['Category'].value_counts()

,count
Category,
Fast Food,1544
Main Course,1539
Desserts,1512
Beverages,1487
Starters,1470
Breads,1445


In [149]:
# invalid Price entries in menu dataset
(clean_datasets['menu']['Price']<0).value_counts()

,count
Price,
False,8907
True,90


In [150]:
clean_datasets['menu'][clean_datasets['menu']['Price']<0]

,FoodItemID,RestaurantID,FoodName,Category,Price,PreparationTime,Calories,Availability
209,210,28,Chicken Burger,Fast Food,-274,11,379.0,Yes
487,488,66,Gulab Jamun,Desserts,-213,7,466.0,Yes
525,526,71,Butter Naan,Breads,-231,26,498.0,Yes
562,563,77,Ice Cream,Desserts,-276,20,414.0,Yes
602,603,83,Rasmalai,Desserts,-291,21,214.0,Yes
...,...,...,...,...,...,...,...,...
8283,8284,1105,Fresh Lime Soda,Beverages,-113,30,413.0,Yes
8538,8539,1139,Chicken Burger,Fast Food,-134,12,516.0,Yes
8638,8639,1153,Veg Manchurian,Starters,-177,19,409.0,Yes
8653,8654,1155,Rasmalai,Desserts,-111,13,557.0,Yes


In [151]:
clean_datasets['order_items'].sort_values('FoodItemID')

,OrderItemID,OrderID,FoodItemID,Quantity,UnitPrice,TotalPrice
26838,26839,113078,1,3,220,660.0
33780,33781,116480,1,2,220,440.0
24341,24342,111889,1,1,220,220.0
14625,14626,107139,1,3,220,660.0
14627,14628,107139,1,1,220,220.0
...,...,...,...,...,...,...
8810,8811,104329,8997,1,234,234.0
9357,9358,104587,8997,3,234,702.0
26544,26545,112937,8997,3,234,702.0
26545,26546,112937,8997,1,234,234.0


In [152]:
# identify number of unique values in UnitPrice per FoodItemID
price_variations = (
    clean_datasets['order_items']
    .groupby('FoodItemID')['UnitPrice']
    .nunique()
)

In [153]:
# checking if there are any more than one unique UnitPrice values
price_variations[price_variations > 1]

,UnitPrice
FoodItemID,


In [154]:
# listing down all the rows with negative Price entries in the menu dataset
negative_prices = clean_datasets['menu'][clean_datasets['menu']['Price']<0][['FoodItemID', 'Price']]

In [155]:
# merging order_items dataset with negative_prices on FoodItemID
price_compare = negative_prices.merge(
    clean_datasets['order_items'][['FoodItemID', 'UnitPrice']].drop_duplicates(),
    on = 'FoodItemID',
    how = 'left'
)

In [156]:
# identifying where UnitPrice from order_items dataset is equal to the abs value of Price in the menu dataset
price_compare['Price_Compare'] = price_compare['UnitPrice'] == price_compare['Price'].abs()

In [157]:
# identifying which FoodItemID have its UnitPrice and Price value unmatched
price_compare[price_compare['Price_Compare'] == False]

,FoodItemID,Price,UnitPrice,Price_Compare
68,6772,-212,NaN,False


In [158]:
# creating supported ids for implementing the abs value in the Price column in the menu dataset
support_ids = price_compare.loc[price_compare['Price_Compare'] == True,'FoodItemID']
support_ids

,FoodItemID
0,210
1,488
2,526
3,563
4,603
...,...
85,8284
86,8539
87,8639
88,8654


In [159]:
# changing the negative Price values in the menu dataset where supported by order_items dataset
clean_datasets['menu'].loc[clean_datasets['menu']['FoodItemID'].isin(support_ids),'Price'] = clean_datasets['menu']['Price'].abs()

In [160]:
cleaning_decisions(
    dataset = 'menu',
    column = 'Price',
    issue = 'negative Price entries',
    evidence = 'there are 90 rows where the Price is negative',
    action = 'converted the negative Price entries to abs where supported by order_items dataset',
    reason = 'cannot retain invalid Price entries',
    before_count = '90',
    changes_count = '89',
    after_count = '1',
    status = 'Completed'
)

In [161]:
# converting the unsupported negative entry in the Price column to NaN
clean_datasets['menu'].loc[clean_datasets['menu']['FoodItemID'] == 6772,'Price'] = np.nan

In [162]:
cleaning_decisions(
    dataset = 'menu',
    column = 'Price',
    issue = 'unsupported negative entry',
    evidence = 'the FoodItemID - 6772 have an unsupported negative entry',
    action = 'converted to NaN',
    reason = 'cannot retain invalid values',
    before_count = '1',
    changes_count = '1',
    after_count = '0',
    status = 'Completed'
)

In [163]:
# checking any negative values in the Calories column
(clean_datasets['menu']['Calories']<0).value_counts()

,count
Calories,
False,8997


In [164]:
cleaning_decisions(
    dataset = 'menu',
    column = 'Calories',
    issue = 'missing Calories entries',
    evidence = 'there are 132 missing entries',
    action = 'retained',
    reason = 'cannot fabricate data',
    before_count = '132',
    changes_count = '0',
    after_count = '132',
    status = 'Profilied'
)

In [165]:
# checking the unique entries in the Availability column
clean_datasets['menu']['Availability'].value_counts()

,count
Availability,
Yes,8060
No,937


In [166]:
column_description(clean_datasets['menu'])

Column: FoodItemID
**************************************************
Description:

count    8997.00000
mean     4499.00000
std      2597.35452
min         1.00000
25%      2250.00000
50%      4499.00000
75%      6748.00000
max      8997.00000
Name: FoodItemID, dtype: float64
Column: RestaurantID
**************************************************
Description:

count    8997.000000
mean      601.390130
std       346.237026
min         1.000000
25%       302.000000
50%       602.000000
75%       901.000000
max      1200.000000
Name: RestaurantID, dtype: float64
Column: FoodName
**************************************************
Description:

count              8997
unique               31
top       Tandoori Roti
freq                384
Name: FoodName, dtype: object
Column: Category
**************************************************
Description:

count          8997
unique            6
top       Fast Food
freq           1544
Name: Category, dtype: object
Column: Price
********************

delivery_partners dataset cleaning decisions

In [167]:
dataset_profiling(clean_datasets['delivery_partners'])

Rows: 2000
Columns: 10
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,DeliveryPartnerID,0,0,int64,2000
1,Name,0,7,object,1993
2,Age,0,1971,int64,29
3,Gender,0,1998,object,2
4,VehicleType,0,1996,object,4
5,JoiningDate,0,249,object,1751
6,City,0,1975,object,25
7,Rating,34,1976,float64,23
8,CompletedDeliveries,0,1111,int64,889
9,AverageDeliveryTime,0,1572,float64,428


In [168]:
clean_datasets['delivery_partners'].head()

,DeliveryPartnerID,Name,Age,Gender,VehicleType,JoiningDate,City,Rating,CompletedDeliveries,AverageDeliveryTime
0,1,Niharika Sathe,37,Male,Bike,22/10/2024,Ahmedabad,3.6,539,35.7
1,2,Zashil Rajagopalan,37,Female,Car,28/11/2023,Lucknow,4.4,978,30.8
2,3,Harish Shan,29,Male,Scooter,25-08-2021,Indore,4.6,518,30.2
3,4,aashi kulkarni,22,Female,Bike,19 Jul 2021,Delhi,4.8,225,47.6
4,5,Maanav Grewal,23,Male,Bike,10/08/2023,Bhubaneswar,4.2,632,-35.7


In [169]:
clean_datasets['delivery_partners']['Name'].str.title().nunique()

1993

In [170]:
# standardising casing across the Name column
clean_datasets['delivery_partners']['Name'] = clean_datasets['delivery_partners']['Name'].str.title()

In [171]:
# checking invalid range values in the Age column
(clean_datasets['delivery_partners']['Age']<0).value_counts()

,count
Age,
False,2000


In [172]:
# checking unique entries in the VehicleType column
clean_datasets['delivery_partners']['VehicleType'].value_counts()

,count
VehicleType,
Bike,1126
Scooter,491
Bicycle,192
Car,191


In [173]:
# analysing JoiningDate column for valid and ambiguous date formats
delivery_partner_dates = date_checks(clean_datasets['delivery_partners'],'DeliveryPartnerID','JoiningDate')
for validity, ids in delivery_partner_dates.items():
  print(f'{validity} count: {len(ids)}')

invalid count: 0
valid_all count: 1016
valid_slash count: 657
valid_dot count: 0
valid_hyphen count: 327
valid_space count: 0
ambiguous_slash count: 237
ambiguous_dot count: 0
ambiguous_hyphen count: 0
ambiguous_space count: 0
formats count: 1763


In [174]:
# parsing all the unambiguous JoiningDate entries and converting the ambiguous JoiningDate entries to NaT retaining the original entries as Raw
clean_datasets['delivery_partners'] = clean_dates(clean_datasets['delivery_partners'],'DeliveryPartnerID','JoiningDate')

In [175]:
cleaning_decisions(
    dataset = 'delivery_partners',
    column = 'JoiningDate',
    issue = 'multiple date formats mixed DD/MM and MM/DD',
    evidence = '237 date values are ambiguous where we cannot determine if the date format is DD/MM or MM/DD',
    action = 'converted the ambiguous Date formats to NaT',
    reason = 'cannot assume the Date values with multiple formats across the column',
    before_count = '237',
    changes_count = '237',
    after_count = '0',
    status = 'Completed'
)


In [176]:
# checking any invalid entries are present in AverageDeliveryTime column
(clean_datasets['delivery_partners']['AverageDeliveryTime']<0).value_counts()

,count
AverageDeliveryTime,
False,1973
True,27


In [177]:
# converting negative values in AverageDeliveryTime column to NaN
clean_datasets['delivery_partners'].loc[clean_datasets['delivery_partners']['AverageDeliveryTime']<0,'AverageDeliveryTime'] = np.nan

In [178]:
cleaning_decisions(
    dataset = 'delivery_partners',
    column = 'AverageDeliveryTime',
    issue = 'negative AverageDeliveryTime entries',
    evidence = 'there are 27 negative entries',
    action = 'converted to NaN',
    reason = 'cannot retain invalid values and cannot fabricate unsupported data',
    before_count = '27',
    changes_count = '27',
    after_count = '0',
    status = 'Completed'
)


In [179]:
# checking values in Rating column
clean_datasets['delivery_partners']['Rating'].isna().value_counts()

,count
Rating,
False,1966
True,34


In [180]:
cleaning_decisions(
    dataset = 'delivery_partners',
    column = 'Rating',
    issue = 'missing',
    evidence = '34 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '34',
    changes_count = '0',
    after_count = '34',
    status = 'Profiled'
)

In [181]:
(clean_datasets['delivery_partners']['City'].isin(clean_datasets['cities']['City'])).value_counts()

,count
City,
True,2000


In [182]:
column_description(clean_datasets['delivery_partners'])

Column: DeliveryPartnerID
**************************************************
Description:

count    2000.000000
mean     1000.500000
std       577.494589
min         1.000000
25%       500.750000
50%      1000.500000
75%      1500.250000
max      2000.000000
Name: DeliveryPartnerID, dtype: float64
Column: Name
**************************************************
Description:

count                 2000
unique                1993
top       Yashvi Venkatesh
freq                     2
Name: Name, dtype: object
Column: Age
**************************************************
Description:

count    2000.000000
mean       27.637500
std         5.596137
min        18.000000
25%        24.000000
50%        28.000000
75%        31.000000
max        47.000000
Name: Age, dtype: float64
Column: Gender
**************************************************
Description:

count     2000
unique       2
top       Male
freq      1003
Name: Gender, dtype: object
Column: VehicleType
******************************

promotions dataset cleaning decisions

In [183]:
dataset_profiling(clean_datasets['promotions'])

Rows: 300
Columns: 6
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,PromotionID,0,0,int64,300
1,CouponCode,0,0,object,300
2,DiscountPercentage,0,293,int64,7
3,CampaignName,0,288,object,12
4,StartDate,0,11,object,289
5,EndDate,0,13,object,287


In [184]:
# checking invalid DiscountPrecentage
(clean_datasets['promotions']['DiscountPercentage']>100).value_counts()

,count
DiscountPercentage,
False,300


In [185]:
# analysing date formats in the StartDate column
promotions_dates = date_checks(clean_datasets['promotions'],'PromotionID','StartDate')

for validity, ids in promotions_dates.items():
  print(f'{validity} count: {len(ids)}')


invalid count: 0
valid_all count: 122
valid_slash count: 117
valid_dot count: 0
valid_hyphen count: 61
valid_space count: 0
ambiguous_slash count: 50
ambiguous_dot count: 0
ambiguous_hyphen count: 0
ambiguous_space count: 0
formats count: 250


In [186]:
# parsing unambiguous dates and converting ambiguous dates to NaT
clean_datasets['promotions'] = clean_dates(clean_datasets['promotions'],'PromotionID','StartDate')

In [187]:
cleaning_decisions(
    dataset = 'promotions',
    column = 'StartDate',
    issue = 'multiple date formats mixed DD/MM and MM/DD',
    evidence = '50 date values are ambiguous where we cannot determine if the date format is DD/MM or MM/DD',
    action = 'converted the ambiguous Date formats to NaT',
    reason = 'cannot assume the Date values with multiple formats across the column',
    before_count = '50',
    changes_count = '50',
    after_count = '0',
    status = 'Completed'
)

In [188]:
# analysing date formats in the EndDate column
promotions_end_dates = date_checks(clean_datasets['promotions'],'PromotionID','EndDate')

for validity, ids in promotions_end_dates.items():
  print(f'{validity} count: {len(ids)}')

invalid count: 0
valid_all count: 121
valid_slash count: 118
valid_dot count: 0
valid_hyphen count: 61
valid_space count: 0
ambiguous_slash count: 39
ambiguous_dot count: 0
ambiguous_hyphen count: 0
ambiguous_space count: 0
formats count: 261


In [189]:
# parsing unambiguous dates and converting ambiguous dates to NaT
clean_datasets['promotions'] = clean_dates(clean_datasets['promotions'],'PromotionID','EndDate')

In [190]:
cleaning_decisions(
    dataset = 'promotions',
    column = 'EndDate',
    issue = 'multiple date formats mixed DD/MM and MM/DD',
    evidence = '39 date values are ambiguous where we cannot determine if the date format is DD/MM or MM/DD',
    action = 'converted the ambiguous Date formats to NaT',
    reason = 'cannot assume the Date values with multiple formats across the column',
    before_count = '39',
    changes_count = '39',
    after_count = '0',
    status = 'Completed'
)

In [191]:
# checking if start date is starting before end date

def check_invalid_date_ranges(dataset, start_date_col, end_date_col):
    """
    Check how many records have StartDate occurring after EndDate.

    Parameters
    ----------
    dataset : pandas.DataFrame
        Dataset containing the start and end date columns.

    start_date_col : str
        Name of the start date column.

    end_date_col : str
        Name of the end date column.

    Returns
    -------
    dict
        Summary containing:
        - total_rows
        - invalid_date_ranges
        - valid_date_ranges
        - missing_date_ranges
        - invalid_percentage
        - invalid_rows
    """

    df = dataset.copy()

    # Convert columns to datetime for comparison.
    # Invalid/unparseable values become NaT.
    start_dates = pd.to_datetime(df[start_date_col], errors='coerce')
    end_dates = pd.to_datetime(df[end_date_col], errors='coerce')

    # Missing / unparseable dates
    missing_dates = start_dates.isna() | end_dates.isna()

    # Start date occurring after end date
    invalid_range = (
        start_dates.notna() &
        end_dates.notna() &
        (start_dates > end_dates)
    )

    # Valid ranges
    valid_range = (
        start_dates.notna() &
        end_dates.notna() &
        (start_dates <= end_dates)
    )

    invalid_count = invalid_range.sum()
    valid_count = valid_range.sum()
    missing_count = missing_dates.sum()

    total_rows = len(df)

    invalid_percentage = (
        invalid_count / total_rows * 100
        if total_rows > 0
        else 0
    )

    # Return the actual problematic rows as well
    invalid_rows = df.loc[
        invalid_range,
        [start_date_col, end_date_col]
    ].copy()

    return {
        'total_rows': total_rows,
        'invalid_date_ranges': invalid_count,
        'valid_date_ranges': valid_count,
        'missing_date_ranges': missing_count,
        'invalid_percentage': invalid_percentage,
        'invalid_rows': invalid_rows
    }

In [192]:
# checking invalid StartDate and EndDate combination in the promotions dataset
result = check_invalid_date_ranges(
    clean_datasets['promotions'],
    'StartDate',
    'EndDate'
)

print("Total rows:", result['total_rows'])
print("Invalid date ranges:", result['invalid_date_ranges'])
print("Valid date ranges:", result['valid_date_ranges'])
print("Missing dates:", result['missing_date_ranges'])
print("Invalid percentage:", result['invalid_percentage'])

Total rows: 300
Invalid date ranges: 4
Valid date ranges: 213
Missing dates: 83
Invalid percentage: 1.3333333333333335


In [193]:
result['invalid_rows']

,StartDate,EndDate
1,2024-02-13,2024-02-09
55,2024-10-21,2024-10-19
160,2023-07-27,2023-07-24
248,2024-08-30,2024-08-25


In [194]:
# convert invalid date ranges to NaT

def convert_invalid_date_ranges_to_nat(dataset, start_date_col, end_date_col):
    """
    Convert date ranges where StartDate > EndDate to NaT.

    Parameters
    ----------
    dataset : pandas.DataFrame
        Dataset containing the start and end date columns.

    start_date_col : str
        Name of the start date column.

    end_date_col : str
        Name of the end date column.

    Returns
    -------
    pandas.DataFrame
        Dataset with invalid date ranges converted to NaT.
    """

    df = dataset.copy()

    # Convert to datetime for comparison
    start_dates = pd.to_datetime(
        df[start_date_col],
        errors='coerce'
    )

    end_dates = pd.to_datetime(
        df[end_date_col],
        errors='coerce'
    )

    # Identify invalid ranges
    invalid_range = (
        start_dates.notna() &
        end_dates.notna() &
        (start_dates > end_dates)
    )

    # Convert invalid dates to NaT
    df.loc[invalid_range, start_date_col] = pd.NaT
    df.loc[invalid_range, end_date_col] = pd.NaT

    return df

In [195]:
clean_datasets['promotions'] = convert_invalid_date_ranges_to_nat(
    clean_datasets['promotions'],
    'StartDate',
    'EndDate'
)

In [196]:
cleaning_decisions(
    dataset = 'promotions',
    column = 'StartDate-EndDate',
    issue = 'StartDate and EndDate combination is invalid',
    evidence = 'there are 4 invalid StartDate and EndDate combinations',
    action = 'converted the invalid date ranges to NaT',
    reason = 'cannot retain invalid entries',
    before_count = '4',
    changes_count = '4',
    after_count = '0',
    status = 'Completed'
)

In [197]:
column_description(clean_datasets['promotions'])

Column: PromotionID
**************************************************
Description:

count    300.000000
mean     150.500000
std       86.746758
min        1.000000
25%       75.750000
50%      150.500000
75%      225.250000
max      300.000000
Name: PromotionID, dtype: float64
Column: CouponCode
**************************************************
Description:

count          300
unique         300
top       ZOMKA2ZX
freq             1
Name: CouponCode, dtype: object
Column: DiscountPercentage
**************************************************
Description:

count    300.000000
mean      27.666667
std       12.842846
min       10.000000
25%       15.000000
50%       25.000000
75%       40.000000
max       50.000000
Name: DiscountPercentage, dtype: float64
Column: CampaignName
**************************************************
Description:

count                    300
unique                    12
top       Mega Cashback Week
freq                      31
Name: CampaignName, dtype: object


restaurants dataset cleaning decisions

In [198]:
dataset_profiling(clean_datasets['restaurants'])

Rows: 1200
Columns: 13
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,RestaurantID,0,0,int64,1200
1,RestaurantName,0,1060,object,140
2,Cuisine,12,1180,object,19
3,City,0,1175,object,25
4,Area,0,1188,object,12
5,OpeningTime,0,1195,object,5
6,ClosingTime,0,1196,object,4
7,Rating,22,1153,float64,46
8,AverageCost,0,607,int64,593
9,OwnerName,0,0,object,1200


In [199]:
cleaning_decisions(
    dataset = 'restaurants',
    column = 'Cuisine',
    issue = 'missing',
    evidence = '12 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '12',
    changes_count = '0',
    after_count = '12',
    status = 'Profiled'
)

In [200]:
cleaning_decisions(
    dataset = 'restaurants',
    column = 'Rating',
    issue = 'missing',
    evidence = '22 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '22',
    changes_count = '0',
    after_count = '22',
    status = 'Profiled'
)

In [201]:
# checking the number of unique RestaurantName entries with standard casing
clean_datasets['restaurants']['RestaurantName'].str.title().nunique()

100

In [202]:
# converting the entries in the RestaurantName column to have standard title casing
clean_datasets['restaurants']['RestaurantName'] = clean_datasets['restaurants']['RestaurantName'].str.title()

In [203]:
cleaning_decisions(
    dataset = 'restaurants',
    column = 'RestaurantName',
    issue = 'inconsisted casing across the entries',
    evidence = 'identified 140 unique values without standard casing and 100 unique values found with standard casing',
    action = 'converted all the entries to have standard title casing',
    reason = 'cannot have duplicate entries considered as unique because of inconsistent casing',
    before_count = '140',
    changes_count = '40',
    after_count = '100',
    status = 'Completed'
)

In [204]:
# checking cities integrity
(clean_datasets['restaurants']['City'].isin(clean_datasets['cities']['City'])).value_counts()

,count
City,
True,1200


In [205]:
# checking number of unique entries with same casing in the Area column
clean_datasets['restaurants']['Area'].str.title().nunique()

12

In [206]:
# checking time formatting across the OpeningTime entries
check_time_formats(clean_datasets['restaurants'],'OpeningTime')

Dataset: OpeningTime
--------------------------------------------------
Valid times   : 1200
Invalid times : 0
Missing times : 0

No invalid time values found.


{'valid_count': 1200,
 'invalid_count': 0,
 'missing_count': 0,
 'invalid_values': []}

In [207]:
# checking time formatting across the ClosingTime entries
check_time_formats(clean_datasets['restaurants'],'ClosingTime')

Dataset: ClosingTime
--------------------------------------------------
Valid times   : 1200
Invalid times : 0
Missing times : 0

No invalid time values found.


{'valid_count': 1200,
 'invalid_count': 0,
 'missing_count': 0,
 'invalid_values': []}

In [208]:
# checking the Rating column entries
clean_datasets['restaurants']['Rating'].head(10)

,Rating
0,3.4
1,NaN
2,4.3
3,3.3
4,3.3
5,3.6
6,3.5
7,4.4
8,3.6
9,3.7


In [209]:
# checking how many entries do not fall in the range of desired Rating
(clean_datasets['restaurants']['Rating']>5).value_counts()

,count
Rating,
False,1182
True,18


In [210]:
(clean_datasets['restaurants']['Rating']<1).value_counts()

,count
Rating,
False,1200


In [211]:
# converting the entries in Rating column which are >5 to NaN
clean_rating(clean_datasets['restaurants'],['Rating'])

Rating: 18 invalid ratings found
Rating is now cleaned with no out of range ratings!


In [212]:
cleaning_decisions(
    dataset = 'restaurants',
    column = 'Rating',
    issue = 'out-of-range range',
    evidence = 'there are 18 entries in the Rating column that are over 5',
    action = 'converted the out-of-range entries to NaN',
    reason = 'cannot retain out-of-range values',
    before_count = '18',
    changes_count = '18',
    after_count = '0',
    status = 'Completed'
)

In [213]:
# checking the invalid range in the AverageCost column
(clean_datasets['restaurants']['AverageCost']<0).value_counts()

,count
AverageCost,
False,1185
True,15


In [214]:
clean_datasets['menu'].head(10)

,FoodItemID,RestaurantID,FoodName,Category,Price,PreparationTime,Calories,Availability
0,1,1,Biryani,Main Course,220.0,16,519.0,Yes
1,2,1,Cheese Pizza,Fast Food,123.0,13,215.0,Yes
2,3,1,Veg Sandwich,Fast Food,40.0,28,440.0,Yes
3,4,1,Butter Naan,Breads,227.0,16,683.0,No
4,5,1,Garlic Naan,Breads,338.0,17,395.0,Yes
5,6,1,Biryani,Main Course,160.0,9,338.0,Yes
6,7,1,Momos,Starters,151.0,3,58.0,No
7,8,1,Kulcha,Breads,174.0,24,284.0,Yes
8,9,2,Fresh Lime Soda,Beverages,337.0,15,272.0,No
9,10,2,Kulcha,Breads,51.0,18,98.0,Yes


In [215]:
# checking if the average cost of all the entries per restaurant in the menu dataset aligns with the valid AverageCost entries in the restaurants column
menu_avg_price = clean_datasets['menu'].groupby('RestaurantID')['Price'].mean().reset_index(name='MeanMenuPrice').astype(int)

In [216]:
clean_datasets['restaurants'].sort_values('RestaurantID').head(10)

,RestaurantID,RestaurantName,Cuisine,City,Area,OpeningTime,ClosingTime,Rating,AverageCost,OwnerName,RestaurantType,Latitude,Longitude
0,1,Curry Cafe,Healthy Food,Vadodara,Civil Lines,09:00,23:00,3.4,328,Qarin Ray,Delivery Only,15.884170,88.525722
1,2,Curry Treats,Burgers,Surat,Market Area,07:00,00:00,NaN,525,Manan Bala,Delivery Only,13.581113,87.706587
2,3,Blue Kitchen,Healthy Food,Nashik,MG Road,07:00,01:00,4.3,171,Nicholas Manne,Cloud Kitchen,22.629797,75.921274
3,4,The Grill,Biryani,Nashik,Station Road,08:00,00:00,3.3,258,Harsh Bhargava,Dine-in & Delivery,16.648774,79.743892
4,5,Golden Diner,Chinese,Pune,Ring Road,11:00,22:00,3.3,529,Daksha Narayanan,Dine-in,30.278253,77.396672
5,6,Green Corner,South Indian,Nagpur,Lake View,07:00,22:00,3.6,500,Guneet Sawhney,Dine-in,31.218729,79.421276
6,7,Golden Grill,Mughlai,Bhubaneswar,Ring Road,07:00,01:00,3.5,381,Leela Mani,Dine-in,24.661424,70.811646
7,8,Green Diner,Desserts,Mumbai,Civil Lines,10:00,01:00,4.4,218,Harish Bobal,Delivery Only,33.790907,77.618196
8,9,Royal Cafe,Chinese,Bhopal,Market Area,09:00,22:00,3.6,533,Vedhika Buch,Cloud Kitchen,14.119996,87.107108
9,10,Green Grill,Desserts,Bhubaneswar,Station Road,10:00,01:00,3.7,487,Gagan Bansal,Delivery Only,9.306706,79.461981


In [217]:
menu_avg_price.head(10)

,RestaurantID,MeanMenuPrice
0,1,179
1,2,179
2,3,175
3,4,186
4,5,166
5,6,168
6,7,123
7,8,136
8,9,215
9,10,139


In [218]:
menu_restaurants_avg_price = menu_avg_price.merge(
    clean_datasets['restaurants'][['RestaurantID','AverageCost']],
    on = 'RestaurantID',
    how = 'left'
)

In [219]:
menu_restaurants_avg_price.head(10)

,RestaurantID,MeanMenuPrice,AverageCost
0,1,179,328
1,2,179,525
2,3,175,171
3,4,186,258
4,5,166,529
5,6,168,500
6,7,123,381
7,8,136,218
8,9,215,533
9,10,139,487


In [220]:
# checking if the average FoodCost per restaurant in the orders dataset aligns with the AverageCost in the restaurants dataset
orders_avg_price = clean_datasets['orders'].groupby('RestaurantID')['FoodCost'].mean().reset_index(name='MeanFoodPrice').astype(int)

In [221]:
orders_avg_price.head(10)

,RestaurantID,MeanFoodPrice
0,1,366
1,2,397
2,3,381
3,4,386
4,5,389
5,6,407
6,7,289
7,8,399
8,9,343
9,10,441


In [222]:
orders_restaurants_avg_price = orders_avg_price.merge(
    clean_datasets['restaurants'][['RestaurantID','AverageCost']],
    on = 'RestaurantID',
    how = 'left'
)

In [223]:
orders_restaurants_avg_price.head()

,RestaurantID,MeanFoodPrice,AverageCost
0,1,366,328.0
1,2,397,525.0
2,3,381,171.0
3,4,386,258.0
4,5,389,529.0


In [224]:
# converting the negative AverageCost entries in the restaurants dataset to NaN
clean_datasets['restaurants'].loc[clean_datasets['restaurants']['AverageCost']<0,'AverageCost'] = np.nan

In [225]:
cleaning_decisions(
    dataset = 'restaurants',
    column = 'AverageCost',
    issue = 'negative AverageCost',
    evidence = 'there are 15 AverageCost entries that are negative',
    action = 'converted the negative entries to NaN',
    reason = 'there is no evidence in other datasets to identify the correct AverageCost entries and hence converted the invalid to NaN',
    before_count = '15',
    changes_count = '15',
    after_count = '0',
    status = 'Completed'
)

In [226]:
# checking the number of unique OwnerName entries with standardised casing
clean_datasets['restaurants']['OwnerName'].str.title().nunique()

1200

In [227]:
# checking the unique RestaurantType entries
clean_datasets['restaurants']['RestaurantType'].value_counts()

,count
RestaurantType,
Cloud Kitchen,251
Dine-in,246
Delivery Only,244
QSR,237
Dine-in & Delivery,222


In [228]:
column_description(clean_datasets['restaurants'])

Column: RestaurantID
**************************************************
Description:

count    1200.000000
mean      600.500000
std       346.554469
min         1.000000
25%       300.750000
50%       600.500000
75%       900.250000
max      1200.000000
Name: RestaurantID, dtype: float64
Column: RestaurantName
**************************************************
Description:

count          1200
unique          100
top       The House
freq             22
Name: RestaurantName, dtype: object
Column: Cuisine
**************************************************
Description:

count             1188
unique              19
top       South Indian
freq                81
Name: Cuisine, dtype: object
Column: City
**************************************************
Description:

count      1200
unique       25
top       Delhi
freq         58
Name: City, dtype: object
Column: Area
**************************************************
Description:

count            1200
unique             12
top       Civil

customers dataset cleaning decisions

In [229]:
dataset_profiling(clean_datasets['customers'])

Rows: 12000
Columns: 13
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,CustomerID,0,0,int64,12000
1,Name,0,251,object,11749
2,Age,168,11915,float64,84
3,Gender,120,11996,object,3
4,Phone,142,141,object,11858
5,Email,328,477,object,11522
6,City,0,11975,object,25
7,State,180,11971,object,28
8,Pincode,0,333,object,11667
9,RegistrationDate,0,6784,object,5216


In [230]:
# checking duplicated Email in customers
clean_datasets['customers'][(clean_datasets['customers']['Email'].duplicated(keep=False)) & (clean_datasets['customers']['Email'].notna())].sort_values('Email')

,CustomerID,Name,Age,Gender,Phone,Email,City,State,Pincode,RegistrationDate,Membership,TotalOrders,PreferredCuisine
7715,7716,JAGAT BHATIA,47.0,Other,1501535342,aadi16@example.org,Nagpur,Tamil Nadu,4175,14 Sep 2022,Gold,3,Street Food
2773,2774,Eiravati Ahluwalia,28.0,Female,7281570662,aadi16@example.org,Visakhapatnam,Tripura,420906,24/06/2023,Basic,4,Mexican
2070,2071,Aarini Brar,NaN,Male,6709144654,abdul00@example.org,Pune,Jharkhand,524075,04/01/2022,Basic,7,Mughlai
2746,2747,Caleb Ghosh,36.0,Male,7457417918,abdul00@example.org,Nagpur,Arunachal Pradesh,180485,15/11/2023,Basic,8,Street Food
848,849,Shivansh Chander,40.0,Male,5014374830,abdulsagar@example.com,Mumbai,Andhra Pradesh,812587,2024-01-19,Basic,6,Mexican
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,8689,Liam Dhar,16.0,Male,7004078000,zbadami@example.org,Guwahati,Bihar,145781,14-02-2024,Gold,5,Desserts
10042,10043,Victor Kant,19.0,Male,2376983884,zlalla@example.com,Kolkata,West Bengal,533407,26-09-2022,Zomato Pro,11,North Indian
4546,4547,Bhavani Jani,17.0,Female,8476505476,zlalla@example.com,Ahmedabad,Punjab,376587,09/15/2024,Basic,7,Healthy Food
10712,10713,Zaid Taneja,31.0,Female,5274345053,zsarna@example.org,Chandigarh,Uttarakhand,415795,08/19/2022,Basic,5,Continental


In [231]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Email',
    issue = 'duplicates',
    evidence = 'there are 297 rows with duplicated Email entries',
    action = 'retained',
    reason = 'cannot identify the single user of the Email and there can be multiple accounts created with same Email and different Phone entries',
    before_count = '297',
    changes_count = '0',
    after_count = '297',
    status = 'Profiled'
)

In [232]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Age',
    issue = 'missing',
    evidence = '168 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '168',
    changes_count = '0',
    after_count = '168',
    status = 'Profiled'
)

In [233]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Gender',
    issue = 'missing',
    evidence = '120 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '120',
    changes_count = '0',
    after_count = '120',
    status = 'Profiled'
)

In [234]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Phone',
    issue = 'missing',
    evidence = '142 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '142',
    changes_count = '0',
    after_count = '142',
    status = 'Profiled'
)

In [235]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Email',
    issue = 'missing',
    evidence = '328 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '328',
    changes_count = '0',
    after_count = '328',
    status = 'Profiled'
)

In [236]:
cleaning_decisions(
    dataset = 'customers',
    column = 'State',
    issue = 'missing',
    evidence = '180 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '180',
    changes_count = '0',
    after_count = '180',
    status = 'Profiled'
)

In [237]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Membership',
    issue = 'missing',
    evidence = '120 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '120',
    changes_count = '0',
    after_count = '120',
    status = 'Profiled'
)

In [238]:
cleaning_decisions(
    dataset = 'customers',
    column = 'PreferredCuisine',
    issue = 'missing',
    evidence = '240 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth',
    before_count = '240',
    changes_count = '0',
    after_count = '240',
    status = 'Profiled'
)

In [239]:
# checking duplicated Phone in customers
clean_datasets['customers'][(clean_datasets['customers']['Phone'].duplicated(keep=False)) & (clean_datasets['customers']['Phone'].notna())].sort_values('Phone')

,CustomerID,Name,Age,Gender,Phone,Email,City,State,Pincode,RegistrationDate,Membership,TotalOrders,PreferredCuisine


In [240]:
# checking if the unique number of Name entries in the customers table changes with standard casing
clean_datasets['customers']['Name'].nunique() - clean_datasets['customers']['Name'].str.title().nunique()

17

In [241]:
clean_datasets['customers']['Name'].str.title().nunique()

11732

In [242]:
# converting all the entries in the Name column to have title casing
clean_datasets['customers']['Name'] = clean_datasets['customers']['Name'].str.title()

In [243]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Name',
    issue = 'inconsisted casing',
    evidence = 'there are 11749 unique Name entries in the customers dataset without standard casing and 17 unique entries changed to duplicates with title casing',
    action = 'converted all the entries to have title casing',
    reason = 'cannot have inconsistent casing across one Column',
    before_count = '11749',
    changes_count = '17',
    after_count = '11732',
    status = 'Completed'
)

In [244]:
# checking negative range values in Age column
(clean_datasets['customers']['Age']<0).value_counts()

,count
Age,
False,11742
True,258


In [245]:
# converting the invalid negative Age entries to NaN
clean_datasets['customers'].loc[clean_datasets['customers']['Age']<0,'Age'] = np.nan

In [246]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Age',
    issue = 'negative Age entries',
    evidence = 'there are 258 negative entries in the Age column',
    action = 'converted negative entries to NaN',
    reason = 'cannot retain invalid entries',
    before_count = '258',
    changes_count = '258',
    after_count = '0',
    status = 'Completed'
)

In [247]:
# checking unique entries in Gender column
clean_datasets['customers']['Gender'].value_counts()

,count
Gender,
Female,3995
Other,3943
Male,3942


In [248]:
# function to check the formatting in Phone column
def check_phone_numbers(dataset, phone_column):
    """
    Check phone numbers for common Indian phone-number data-quality issues.

    The function does NOT modify the dataset.

    Checks:
    - Missing values
    - Empty values
    - +91 country-code format
    - 91 country-code format
    - Formatting characters such as spaces, hyphens, parentheses
    - Non-digit characters
    - Incorrect digit length
    - Invalid Indian mobile starting digit
    - Repeated-digit numbers
    - Duplicate phone numbers

    Returns:
        Dictionary containing summary statistics and problematic values.
    """

    df = dataset.copy()

    # Keep original values as strings
    phone = df[phone_column].astype('string').str.strip()

    # ---------------------------------------------------------
    # 1. Missing / empty values
    # ---------------------------------------------------------

    missing = phone.isna()
    empty = phone.eq('')

    # ---------------------------------------------------------
    # 2. Country-code detection
    # ---------------------------------------------------------

    has_plus91 = phone.str.match(
        r'^\+91',
        na=False
    )

    has_91 = phone.str.match(
        r'^91\d{10}$',
        na=False
    )

    # ---------------------------------------------------------
    # 3. Formatting characters
    # ---------------------------------------------------------

    has_formatting = phone.str.contains(
        r'[\s\-\(\)]',
        regex=True,
        na=False
    )

    # ---------------------------------------------------------
    # 4. Remove common formatting temporarily
    #    ONLY for validation
    # ---------------------------------------------------------

    normalized = (
        phone
        .str.replace(r'[\s\-\(\)]', '', regex=True)
        .str.replace(r'^\+91', '', regex=True)
        .str.replace(r'^91(?=\d{10}$)', '', regex=True)
    )

    # ---------------------------------------------------------
    # 5. Non-digit characters after normalization
    # ---------------------------------------------------------

    non_digit = (
        normalized.notna()
        & ~normalized.str.fullmatch(r'\d+', na=False)
    )

    # ---------------------------------------------------------
    # 6. Digit count
    # ---------------------------------------------------------

    digit_count = normalized.str.replace(
        r'\D',
        '',
        regex=True
    ).str.len()

    wrong_length = (
        normalized.notna()
        & ~empty
        & (digit_count != 10)
    )

    # ---------------------------------------------------------
    # 7. Invalid Indian mobile starting digit
    # ---------------------------------------------------------

    invalid_start = (
        normalized.notna()
        & ~empty
        & (digit_count == 10)
        & ~normalized.str.match(r'^[6-9]', na=False)
    )

    # ---------------------------------------------------------
    # 8. Repeated-digit numbers
    # ---------------------------------------------------------

    repeated_digits = normalized.str.fullmatch(
        r'(\d)\1{9}',
        na=False
    )

    # ---------------------------------------------------------
    # 9. Overall validity
    # ---------------------------------------------------------

    valid = normalized.str.fullmatch(
        r'[5-9]\d{9}',
        na=False
    )

    valid = (
        valid
        & ~repeated_digits
    )

    # ---------------------------------------------------------
    # 10. Duplicate phone numbers
    # ---------------------------------------------------------

    duplicate_phone = (
        normalized.notna()
        & normalized.duplicated(keep=False)
    )

    # ---------------------------------------------------------
    # 11. Summary
    # ---------------------------------------------------------

    summary = {
        'total_rows': len(df),
        'valid_numbers': valid.sum(),
        'missing_numbers': missing.sum(),
        'empty_numbers': empty.sum(),
        'plus_91_numbers': has_plus91.sum(),
        '91_country_code_numbers': has_91.sum(),
        'formatted_numbers': has_formatting.sum(),
        'non_digit_numbers': non_digit.sum(),
        'wrong_length_numbers': wrong_length.sum(),
        'invalid_start_digit': invalid_start.sum(),
        'repeated_digit_numbers': repeated_digits.sum(),
        'duplicate_phone_numbers': duplicate_phone.sum()
    }

    # ---------------------------------------------------------
    # 12. Problematic rows
    # ---------------------------------------------------------

    issues = df.loc[
        (
            missing
            | empty
            | non_digit
            | wrong_length
            | invalid_start
            | repeated_digits
        ),
        [phone_column]
    ].copy()

    return {
        'summary': summary,
        'issues': issues
    }

In [249]:
# checking valid and invalid Phone entries in the customers dataset
check_phone_numbers(clean_datasets['customers'],'Phone')

{'summary': {'total_rows': 12000,
  'valid_numbers': np.int64(5729),
  'missing_numbers': np.int64(142),
  'empty_numbers': np.int64(0),
  'plus_91_numbers': np.int64(0),
  '91_country_code_numbers': np.int64(133),
  'formatted_numbers': np.int64(0),
  'non_digit_numbers': np.int64(160),
  'wrong_length_numbers': np.int64(317),
  'invalid_start_digit': np.int64(6953),
  'repeated_digit_numbers': np.int64(0),
  'duplicate_phone_numbers': np.int64(0)},
 'issues':               Phone
 0        1960013389
 1        2351161559
 2        0341316475
 3        4835030564
 6      914272602734
 ...             ...
 11993    2345161665
 11995    2999545770
 11996    98B242FADG
 11998    5077489383
 11999    4876374264
 
 [7412 rows x 1 columns]}

In [250]:
# function to clean Phone entries
def clean_phone_numbers(dataset, phone_column):
    """
    Clean and standardize Indian mobile phone numbers.

    Transformations:
    - Preserves original values in <phone_column>_raw
    - Removes spaces, hyphens, and parentheses
    - Converts +91XXXXXXXXXX to XXXXXXXXXX
    - Converts 91XXXXXXXXXX to XXXXXXXXXX
    - Converts valid numbers to standardized 10-digit format
    - Converts unresolved invalid numbers to <NA>

    Returns:
        Cleaned DataFrame.
    """

    df = dataset.copy()

    # ---------------------------------------------------------
    # 1. Preserve original values
    # ---------------------------------------------------------

    df[f'{phone_column}_raw'] = df[phone_column]

    # ---------------------------------------------------------
    # 2. Convert to string
    # ---------------------------------------------------------

    phone = df[phone_column].astype('string').str.strip()

    # Empty strings → missing
    phone = phone.replace('', pd.NA)

    # ---------------------------------------------------------
    # 3. Remove common formatting characters
    # ---------------------------------------------------------

    phone = phone.str.replace(
        r'[\s\-\(\)]',
        '',
        regex=True
    )

    # ---------------------------------------------------------
    # 4. Remove Indian country code
    # ---------------------------------------------------------

    phone = phone.str.replace(
        r'^\+91',
        '',
        regex=True
    )

    phone = phone.str.replace(
        r'^91(?=\d{10}$)',
        '',
        regex=True
    )

    # ---------------------------------------------------------
    # 5. Validate standardized number
    # ---------------------------------------------------------

    valid = phone.str.fullmatch(
        r'[5-9]\d{9}',
        na=False
    )

    # Reject obvious repeated-digit values
    repeated_digits = phone.str.fullmatch(
        r'(\d)\1{9}',
        na=False
    )

    valid = valid & ~repeated_digits

    # ---------------------------------------------------------
    # 6. Convert unresolved invalid values to missing
    # ---------------------------------------------------------

    phone[~valid] = pd.NA

    # ---------------------------------------------------------
    # 7. Store cleaned values
    # ---------------------------------------------------------

    df[phone_column] = phone

    return df

In [251]:
# BEFORE cleaning Phone entries in the customers dataset
before_phone_check = check_phone_numbers(
    clean_datasets['customers'],
    'Phone'
)

print(before_phone_check['summary'])

{'total_rows': 12000, 'valid_numbers': np.int64(5729), 'missing_numbers': np.int64(142), 'empty_numbers': np.int64(0), 'plus_91_numbers': np.int64(0), '91_country_code_numbers': np.int64(133), 'formatted_numbers': np.int64(0), 'non_digit_numbers': np.int64(160), 'wrong_length_numbers': np.int64(317), 'invalid_start_digit': np.int64(6953), 'repeated_digit_numbers': np.int64(0), 'duplicate_phone_numbers': np.int64(0)}


In [252]:
clean_datasets['customers'] = clean_phone_numbers(
    clean_datasets['customers'],
    'Phone'
)

In [253]:
after_phone_check = check_phone_numbers(
    clean_datasets['customers'],
    'Phone'
)

print(after_phone_check['summary'])

{'total_rows': 12000, 'valid_numbers': np.int64(5729), 'missing_numbers': np.int64(6271), 'empty_numbers': np.int64(0), 'plus_91_numbers': np.int64(0), '91_country_code_numbers': np.int64(0), 'formatted_numbers': np.int64(0), 'non_digit_numbers': np.int64(0), 'wrong_length_numbers': np.int64(0), 'invalid_start_digit': np.int64(1141), 'repeated_digit_numbers': np.int64(0), 'duplicate_phone_numbers': np.int64(0)}


In [254]:
clean_datasets['customers'].head()

,CustomerID,Name,Age,Gender,Phone,Email,City,State,Pincode,RegistrationDate,Membership,TotalOrders,PreferredCuisine,Phone_raw
0,1,Aryan Maharaj,35.0,Male,<NA>,issaconi@example.org,Nagpur,Haryana,769177,2022-06-11,Basic,4,Fast Food,1960013389
1,2,Arunima Ahuja,29.0,Male,<NA>,caleb78@example.org,Hyderabad,Tripura,776564,18-02-2024,Zomato Pro,6,Mughlai,2351161559
2,3,Farhan Memon,26.0,Other,<NA>,kamalapant@example.org,Vadodara,Uttarakhand,507888,05/17/2023,Gold,5,Mughlai,0341316475
3,4,Ekbal Garg,26.0,Female,<NA>,kakardivya@example.org,Jaipur,Nagaland,439964,2022.05.16,Gold,6,Street Food,4835030564
4,5,Meghana Shanker,32.0,Female,8496965328,wchand@example.org,Vadodara,Chhattisgarh,489581,27/05/2024,Basic,6,Desserts,8496965328


In [255]:
print('Unique Phone entries count before cleaning:', clean_datasets['customers']['Phone_raw'].nunique())
print('Unique Phone entries count after cleaning:', clean_datasets['customers']['Phone'].nunique())

Unique Phone entries count before cleaning: 11858
Unique Phone entries count after cleaning: 5729


In [256]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Phone',
    issue = 'invalid Phone entries',
    evidence = 'there are Phone entries that contain wrong digit count, invalid start numbers, contain letters',
    action = 'converted numbers starting with country code 91 to standard 10 digits, converted invalid entries to NaN and retained original as Raw',
    reason = 'cannot retain invalid Phone entries',
    before_count = '11858',
    changes_count = '6129',
    after_count = '5729',
    status = 'Completed'
)

In [257]:
# function to check Email entries
def check_email_addresses(dataset, email_column):
    """
    Check email addresses for common data-quality issues.

    The function does not modify the dataset.

    Checks:
    - Missing values
    - Empty values
    - Leading/trailing whitespace
    - Uppercase characters
    - Basic email structure
    - Missing @ symbol
    - Invalid email structure
    - Duplicate email addresses

    Returns:
        Dictionary containing summary statistics and problematic values.
    """

    df = dataset.copy()

    email = df[email_column].astype('string')

    missing = email.isna()
    empty = email.str.strip().eq('')

    whitespace = (
        email.notna() &
        (email != email.str.strip())
    )

    uppercase = (
        email.notna() &
        (email != email.str.lower())
    )

    # Basic email structure
    valid_format = email.str.fullmatch(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        na=False
    )

    invalid_format = (
        email.notna() &
        ~empty &
        ~valid_format
    )

    missing_at = (
        email.notna() &
        ~empty &
        ~email.str.contains('@', regex=False, na=False)
    )

    # Normalize temporarily for duplicate checking
    normalized = email.str.strip().str.lower()

    duplicate_email = (
        normalized.notna() &
        normalized.duplicated(keep=False)
    )

    return {
        'summary': {
            'total_rows': len(df),
            'valid_emails': valid_format.sum(),
            'missing_emails': missing.sum(),
            'empty_emails': empty.sum(),
            'leading_trailing_whitespace': whitespace.sum(),
            'uppercase_emails': uppercase.sum(),
            'invalid_format': invalid_format.sum(),
            'missing_at_symbol': missing_at.sum(),
            'duplicate_emails': duplicate_email.sum()
        },

        'invalid_values': df.loc[
            invalid_format,
            [email_column]
        ].drop_duplicates(),

        'duplicate_values': df.loc[
            duplicate_email,
            [email_column]
        ].drop_duplicates()
    }

In [258]:
# function to clean Email entries
def clean_email_addresses(dataset, email_column):
    """
    Clean and standardize email addresses.

    Changes:
    - Preserves original values in <email_column>_raw
    - Removes leading/trailing whitespace
    - Converts emails to lowercase
    - Converts empty values to NaN
    - Converts structurally invalid emails to NaN

    Returns:
        Cleaned DataFrame.
    """

    df = dataset.copy()

    # Preserve original values
    df[f'{email_column}_raw'] = df[email_column]

    email = df[email_column].astype('string')

    # Remove leading/trailing whitespace
    email = email.str.strip()

    # Empty strings → missing
    email = email.replace('', pd.NA)

    # Standardize case
    email = email.str.lower()

    # Validate basic email structure
    valid = email.str.fullmatch(
        r'[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}',
        na=False
    )

    # Invalid values → missing
    email[~valid] = pd.NA

    df[email_column] = email

    return df

In [259]:
# checking for invalid Email entries in the customers dataset
check_email_addresses(clean_datasets['customers'],'Email')

{'summary': {'total_rows': 12000,
  'valid_emails': np.int64(11672),
  'missing_emails': np.int64(328),
  'empty_emails': np.int64(0),
  'leading_trailing_whitespace': np.int64(0),
  'uppercase_emails': np.int64(0),
  'invalid_format': np.int64(0),
  'missing_at_symbol': np.int64(0),
  'duplicate_emails': np.int64(297)},
 'invalid_values': Empty DataFrame
 Columns: [Email]
 Index: [],
 'duplicate_values':                        Email
 87         dsule@example.org
 133      priya29@example.org
 134    rbhargava@example.net
 188      csachar@example.net
 264      nkhanna@example.net
 ...                      ...
 9050      nwable@example.org
 9078     rkarnik@example.org
 9321      fratti@example.com
 9912     wnarain@example.org
 11434    leela86@example.net
 
 [147 rows x 1 columns]}

In [260]:
# function to check Pincode entries
def check_pincodes(dataset, pincode_column):
    """
    Check Indian pincodes for common data-quality issues.

    The function does not modify the dataset.

    Checks:
    - Missing values
    - Empty values
    - Non-digit characters
    - Incorrect length
    - Invalid first digit
    - Duplicate pincodes

    Returns:
        Dictionary containing summary statistics and problematic values.
    """

    df = dataset.copy()

    pincode = df[pincode_column].astype('string').str.strip()

    missing = pincode.isna()
    empty = pincode.eq('')

    non_digit = (
        pincode.notna() &
        ~pincode.str.fullmatch(r'\d+', na=False)
    )

    wrong_length = (
        pincode.notna() &
        ~empty &
        (pincode.str.len() != 6)
    )

    invalid_start = (
        pincode.notna() &
        ~empty &
        pincode.str.fullmatch(r'[1-9]\d{5}', na=False).eq(False)
    )

    valid = pincode.str.fullmatch(
        r'[1-9]\d{5}',
        na=False
    )

    duplicate_pincode = (
        pincode.notna() &
        pincode.duplicated(keep=False)
    )

    return {
        'summary': {
            'total_rows': len(df),
            'valid_pincodes': valid.sum(),
            'missing_pincodes': missing.sum(),
            'empty_pincodes': empty.sum(),
            'non_digit_pincodes': non_digit.sum(),
            'wrong_length': wrong_length.sum(),
            'invalid_start_digit': invalid_start.sum(),
            'duplicate_pincodes': duplicate_pincode.sum()
        },

        'invalid_values': df.loc[
            ~valid & ~missing & ~empty,
            [pincode_column]
        ].drop_duplicates(),

        'duplicate_values': df.loc[
            duplicate_pincode,
            [pincode_column]
        ].drop_duplicates()
    }

In [261]:
# function to clean Pincode entries
def clean_pincodes(dataset, pincode_column):
    """
    Clean and standardize Indian pincodes.

    Changes:
    - Preserves original values in <pincode_column>_raw
    - Removes leading/trailing whitespace
    - Converts numeric-looking values to string
    - Converts valid pincodes to 6-digit format
    - Converts unresolved invalid values to NaN

    Returns:
        Cleaned DataFrame.
    """

    df = dataset.copy()

    # Preserve original values
    df[f'{pincode_column}_raw'] = df[pincode_column]

    pincode = df[pincode_column].astype('string').str.strip()

    # Empty strings → missing
    pincode = pincode.replace('', pd.NA)

    # Validate
    valid = pincode.str.fullmatch(
        r'[1-9]\d{5}',
        na=False
    )

    # Invalid values → missing
    pincode[~valid] = pd.NA

    df[pincode_column] = pincode

    return df

In [262]:
# checking for invalid entries in the Pincode column of customers dataset
check_pincodes(clean_datasets['customers'],'Pincode')

{'summary': {'total_rows': 12000,
  'valid_pincodes': np.int64(11480),
  'missing_pincodes': np.int64(0),
  'empty_pincodes': np.int64(0),
  'non_digit_pincodes': np.int64(254),
  'wrong_length': np.int64(258),
  'invalid_start_digit': np.int64(520),
  'duplicate_pincodes': np.int64(425)},
 'invalid_values':        Pincode
 55      85529X
 61      86931X
 73        7551
 142    -110001
 174     82642X
 ...        ...
 11934     8931
 11950   93388X
 11956      737
 11976   14598X
 11982   26457X
 
 [276 rows x 1 columns],
 'duplicate_values':        Pincode
 62      307817
 129     480038
 142    -110001
 284     486106
 364     429589
 ...        ...
 9109    426258
 9538    308731
 10201   608578
 10501   505444
 10771   652601
 
 [92 rows x 1 columns]}

In [263]:
# cleaning Pincode entries in the customers dataset
clean_datasets['customers'] = clean_pincodes(
    clean_datasets['customers'],
    'Pincode'
)

In [264]:
(clean_datasets['customers']['Pincode'].isna()).value_counts()

,count
Pincode,
False,11480
True,520


In [265]:
clean_datasets['customers']['Pincode'].nunique()

11391

In [266]:
print('number of unique Pincode entries before cleaning:',clean_datasets['customers']['Pincode_raw'].nunique())
print('number of unique Pincode entries after cleaning:',clean_datasets['customers']['Pincode'].nunique())

number of unique Pincode entries before cleaning: 11667
number of unique Pincode entries after cleaning: 11391


In [267]:
cleaning_decisions(
    dataset = 'customers',
    column = 'Pincode',
    issue = 'invalid Pincode entries',
    evidence = 'there are invalin Pincode entries that have letters and invalid start value',
    action = 'converted the invalid entries to NaT',
    reason = 'cannot retain invalid entries',
    before_count = '11667',
    changes_count = '276',
    after_count = '11391',
    status = 'Completed'
)

In [268]:
# checking cities integrity
(clean_datasets['customers']['City'].isin(clean_datasets['cities']['City'])).value_counts()

,count
City,
True,12000


In [269]:
# checking unique number of entries in State column
clean_datasets['customers']['State'].nunique()

28

In [270]:
# checking unique number of entries in State column with standardised casing
clean_datasets['customers']['State'].str.title().nunique()

28

In [271]:
# grouping cities to identify how many unique State entries per City exist
city_state = clean_datasets['customers'].groupby('City')['State'].nunique()

In [272]:
city_state

,State
City,
Ahmedabad,28
Bengaluru,28
Bhopal,28
Bhubaneswar,28
Chandigarh,28
Chennai,28
Coimbatore,28
Delhi,28
Guwahati,28


In [273]:
cleaning_decisions(
    dataset = 'customers',
    column = 'City-State',
    issue = 'each city is mapped with multiple states',
    evidence = 'all 25 cities listed are mapped with all 28 unique states listed in the customers dataset',
    action = 'retain',
    reason = 'no dataset leading to map the correct state of the customer',
    before_count = '12000',
    changes_count = '0',
    after_count = '12000',
    status = 'retainged'
)


In [274]:
# checking RegistrationDate entries in the customers dataset
clean_datasets['customers']['RegistrationDate'].head(10)

,RegistrationDate
0,2022-06-11
1,18-02-2024
2,05/17/2023
3,2022.05.16
4,27/05/2024
5,28 Feb 2024
6,08/25/2023
7,2024.12.22
8,2024.09.22
9,16-02-2023


In [275]:
# analysing date formats in the RegistrationDate column in the customers dataset
customer_dates = date_checks(clean_datasets['customers'],'CustomerID','RegistrationDate')
for validity, ids in customer_dates.items():
  print(f'{validity} count: {len(ids)}')


invalid count: 0
valid_all count: 6034
valid_slash count: 3975
valid_dot count: 0
valid_hyphen count: 1991
valid_space count: 0
ambiguous_slash count: 1420
ambiguous_dot count: 0
ambiguous_hyphen count: 0
ambiguous_space count: 0
formats count: 10580


In [276]:
# parsing the valid and unambiguous RegistrationDate entries
clean_datasets['customers'] = clean_dates(clean_datasets['customers'],'CustomerID','RegistrationDate')

In [277]:
cleaning_decisions(
    dataset = 'customers',
    column = 'RegistrationDate',
    issue = 'multiple date formats mixed DD/MM and MM/DD',
    evidence = '1420 date values are ambiguous where we cannot determine if the date format is DD/MM or MM/DD',
    action = 'converted the ambiguous Date formats to NaT',
    reason = 'cannot assume the Date values with multiple formats across the column',
    before_count = '1420',
    changes_count = '1420',
    after_count = '0',
    status = 'Completed'
)


In [278]:
# checking if there are any negative entries in the TotalOrders column
(clean_datasets['customers']['TotalOrders']<0).value_counts()

,count
TotalOrders,
False,12000


In [279]:
clean_datasets['customers']['Gender'].value_counts()

,count
Gender,
Female,3995
Other,3943
Male,3942


In [280]:
clean_datasets['customers']['Membership'].value_counts()

,count
Membership,
Basic,7063
Zomato Pro,2449
Gold,2368


In [281]:
clean_datasets['customers']['PreferredCuisine'].value_counts()

,count
PreferredCuisine,
North Indian,878
Desserts,877
Mughlai,873
Mexican,860
Thai,845
Bakery,843
South Indian,840
Fast Food,835
Street Food,835


In [282]:
column_description(clean_datasets['customers'])

Column: CustomerID
**************************************************
Description:

count    12000.00000
mean      6000.50000
std       3464.24595
min          1.00000
25%       3000.75000
50%       6000.50000
75%       9000.25000
max      12000.00000
Name: CustomerID, dtype: float64
Column: Name
**************************************************
Description:

count                  12000
unique                 11732
top       Harini Subramaniam
freq                       3
Name: Name, dtype: object
Column: Age
**************************************************
Description:

count    11574.000000
mean        30.760670
std          8.550545
min         16.000000
25%         25.000000
50%         30.000000
75%         37.000000
max         66.000000
Name: Age, dtype: float64
Column: Gender
**************************************************
Description:

count      11880
unique         3
top       Female
freq        3995
Name: Gender, dtype: object
Column: Phone
**************************

orders dataset cleaning decisions

In [283]:
dataset_profiling(clean_datasets['orders'])

Rows: 20500
Columns: 15
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,OrderID,0,0,int64,20500
1,CustomerID,0,10659,int64,9841
2,RestaurantID,0,19287,int64,1213
3,DeliveryPartnerID,0,18500,int64,2000
4,OrderDate,0,17149,object,3351
5,OrderTime,0,19075,object,1425
6,DeliveryTimeMinutes,0,20254,int64,246
7,FoodCost,0,19555,int64,945
8,DeliveryFee,0,20494,int64,6
9,Discount,0,18337,float64,2163


In [284]:
cleaning_decisions(
    dataset = 'orders',
    column = 'CouponCode',
    issue = 'missing',
    evidence = '13281 missing values found',
    action = 'retain',
    reason = 'cannot fabricate data without truth and there can be orders without using any CouponCode',
    before_count = '13281',
    changes_count = '0',
    after_count = '13281',
    status = 'Profiled'
)

In [285]:
# foreign key integrity checks
(clean_datasets['orders']['RestaurantID'].isin(clean_datasets['restaurants']['RestaurantID'])).value_counts()

,count
RestaurantID,
True,20487
False,13


In [286]:
(clean_datasets['orders']['CustomerID'].isin(clean_datasets['customers']['CustomerID'])).value_counts()

,count
CustomerID,
True,20488
False,12


In [287]:
(clean_datasets['orders']['DeliveryPartnerID'].isin(clean_datasets['delivery_partners']['DeliveryPartnerID'])).value_counts()

,count
DeliveryPartnerID,
True,20500


In [288]:
# checking orders without existing CustomerID
orphan_customers = (
    clean_datasets['orders']['CustomerID'].notna()
    & ~clean_datasets['orders']['CustomerID'].isin(
        clean_datasets['customers']['CustomerID']
    )
)
# checking orders without existing RestaurantID
orphan_restaurants = (
    clean_datasets['orders']['RestaurantID'].notna()
    & ~clean_datasets['orders']['RestaurantID'].isin(
        clean_datasets['restaurants']['RestaurantID']
    )
)

In [289]:
# check if the OrderID of the orphan CustomerID entries are listed in CustomerReview dataset
(clean_datasets['orders'].loc[orphan_customers, 'OrderID'].isin(clean_datasets['customer_feedback']['OrderID'])).value_counts()

,count
OrderID,
True,9
False,3


In [290]:
# check if the OrderID of the orphan RestaurantID entries are listed in CustomerReview dataset
(clean_datasets['orders'].loc[orphan_restaurants, 'OrderID'].isin(clean_datasets['customer_feedback']['OrderID'])).value_counts()

,count
OrderID,
True,11
False,2


In [291]:
# converting the CustomerID entry to NA where no truth can be found in the customers dataset
clean_datasets['orders'].loc[orphan_customers, 'CustomerID'] = pd.NA
# converting the RestaurantID entry to NA where no truth can be found in the restaurants dataset
clean_datasets['orders'].loc[orphan_restaurants, 'RestaurantID'] = pd.NA

In [292]:
cleaning_decisions(
    dataset = 'orders',
    column = 'CustomerID',
    issue = 'non-existent CustomerID in the customers dataset',
    evidence = 'there are 12 orphan CustomerID entries in the orders dataset',
    action = 'coverted the 12 orphan CustomerID to NA',
    reason = 'cannot retain invalid foreign key information but have to retain the order information associated with it',
    before_count = '12',
    changes_count = '12',
    after_count = '0',
    status = 'Completed'
)

In [293]:
cleaning_decisions(
    dataset = 'orders',
    column = 'RestaurantID',
    issue = 'non-existent RestaurantID in the customers dataset',
    evidence = 'there are 13 orphan RestaurantID entries in the orders dataset',
    action = 'coverted the 13 orphan RestaurantID to NA',
    reason = 'cannot retain invalid foreign key information but have to retain the order information associated with it',
    before_count = '13',
    changes_count = '13',
    after_count = '0',
    status = 'Completed'
)

In [294]:
# analysing date formats in the OrderDate column in the orders dataset
order_dates = date_checks(clean_datasets['orders'],'OrderID','OrderDate')
for validity, ids in order_dates.items():
  print(f'{validity} count: {len(ids)}')


invalid count: 0
valid_all count: 8227
valid_slash count: 8111
valid_dot count: 0
valid_hyphen count: 4162
valid_space count: 0
ambiguous_slash count: 2913
ambiguous_dot count: 0
ambiguous_hyphen count: 0
ambiguous_space count: 0
formats count: 17587


In [295]:
clean_datasets['orders']['OrderDate'].isna().value_counts()

,count
OrderDate,
False,20500


In [296]:
# parsing the mixed date formats in the OrderDate column
clean_datasets['orders'] = clean_dates(clean_datasets['orders'],'OrderID','OrderDate')

In [297]:
cleaning_decisions(
    dataset = 'orders',
    column = 'OrderDate',
    issue = 'multiple date formats mixed DD/MM and MM/DD',
    evidence = '2913 date values are ambiguous where we cannot determine if the date format is DD/MM or MM/DD',
    action = 'converted the ambiguous Date formats to NaT',
    reason = 'cannot assume the Date values with multiple formats across the column',
    before_count = '2913',
    changes_count = '2913',
    after_count = '0',
    status = 'Completed'
)

In [298]:
# checking time formats in the OrderTime column of the orders dataset
check_time_formats(clean_datasets['orders'],'OrderTime')

Dataset: OrderTime
--------------------------------------------------
Valid times   : 20500
Invalid times : 0
Missing times : 0

No invalid time values found.


{'valid_count': 20500,
 'invalid_count': 0,
 'missing_count': 0,
 'invalid_values': []}

In [299]:
# checking any negative entries in the DeliveryTimeMinutes columns
(clean_datasets['orders']['DeliveryTimeMinutes']<0).value_counts()

,count
DeliveryTimeMinutes,
False,20292
True,208


In [300]:
# converting the negative entries in the DeliveryTimeMinutes column to NA
clean_datasets['orders'].loc[clean_datasets['orders']['DeliveryTimeMinutes']<0,'DeliveryTimeMinutes'] = pd.NA

In [301]:
cleaning_decisions(
    dataset = 'orders',
    column = 'DeliveryTimeMinutes',
    issue = 'negative DeliveryTimeMinutes entries',
    evidence = 'there are 208 negative entries in the DeliveryTimeMinutes column',
    action = 'converted the negative entries in the DeliveryTimeMinutes column to NA',
    reason = 'cannot retain invalid data and cannot assume the actual data',
    before_count = '208',
    changes_count = '208',
    after_count = '0',
    status = 'Completed'
)

In [302]:
# checking outliers in the DeliveryTimeMinutes
clean_datasets['orders']['DeliveryTimeMinutes'].describe()

,DeliveryTimeMinutes
count,20292.000000
mean,38.587719
std,26.279812
min,8.000000
25%,28.000000
50%,35.000000
75%,43.000000
max,399.000000


In [303]:
(clean_datasets['orders']['DeliveryTimeMinutes']<120).value_counts()

,count
DeliveryTimeMinutes,
True,20147
False,353


In [304]:
clean_datasets['orders'].loc[clean_datasets['orders']['DeliveryTimeMinutes']<120,'DeliveryTimeMinutes'].max()

106.0

In [305]:
clean_datasets['orders'].loc[clean_datasets['orders']['DeliveryTimeMinutes']>120,'DeliveryTimeMinutes'].sort_values()

,DeliveryTimeMinutes
4537,200.0
11466,200.0
7021,201.0
581,201.0
12311,201.0
...,...
20196,392.0
3477,394.0
2385,397.0
15701,398.0


In [306]:
clean_datasets['orders'].loc[clean_datasets['orders']['DeliveryTimeMinutes']>120,'DeliveryTimeMinutes'] = pd.NA

In [307]:
cleaning_decisions(
    dataset = 'orders',
    column = 'DeliveryTimeMinutes',
    issue = 'outliers >120 minutes leading to 399 minutes',
    evidence = 'there are 145 entries with DeliveryTimeMinutes > 120 minutes',
    action = 'converted DeliveryTimeMinutes > 120 to NA',
    reason = 'cannot retain possible outliers in the data also there are no entries from 106 to 200 minutes',
    before_count = '145',
    changes_count = '145',
    after_count = '0',
    status = 'Completed'
)


In [308]:
# checking any negative FoodCost values in the orders dataset
(clean_datasets['orders']['FoodCost']<0).value_counts()

,count
FoodCost,
False,20319
True,181


In [309]:
clean_datasets['order_items'].sort_values('OrderID').head(10)

,OrderItemID,OrderID,FoodItemID,Quantity,UnitPrice,TotalPrice
0,1,100001,3539,3,194,582.0
1,2,100001,3539,1,194,194.0
2,3,100002,6183,3,238,714.0
3,4,100002,6182,1,371,371.0
4,5,100003,771,1,114,114.0
5,6,100004,6918,4,91,364.0
6,7,100005,7047,1,358,358.0
7,8,100005,7045,3,178,534.0
9,10,100006,6,1,160,160.0
8,9,100006,8,1,174,174.0


In [310]:
# grouping order_items dataset by OrderID and summing the TotalPrice column
order_item_cost = (
    clean_datasets['order_items']
    .groupby('OrderID')['TotalPrice']
    .sum()
    .reset_index(name='OrderItemTotalPrice')
)

In [311]:
# merging orders dataset with the total cost per order in the order_items dataset
food_cost_check = clean_datasets['orders'].merge(
    order_item_cost,
    on='OrderID',
    how='left'
)

In [312]:
# checking if the abs values in the FoodCost column in orders dataset match with the calculated TotalPrice per order in the order_items dataset
food_cost_check['FoodCostMatch'] = (
    food_cost_check['FoodCost'].abs()
    == food_cost_check['OrderItemTotalPrice'].abs()
)

In [313]:
food_cost_check.sort_values('FoodCostMatch')

,OrderID,CustomerID,RestaurantID,DeliveryPartnerID,OrderDate,OrderTime,DeliveryTimeMinutes,FoodCost,DeliveryFee,Discount,CouponCode,GST,FinalAmount,OrderStatus,PaymentMethod,OrderItemTotalPrice,FoodCostMatch
0,100001,7095.0,474.0,121,2023-06-17,11:51:00,43.0,206,40,103.0,ZOMKXZZW,10.30,153.30,Delivered,Cash on Delivery,776.0,False
13668,113669,5485.0,707.0,494,2023-10-20,15:54:00,70.0,251,49,0.0,NaN,12.55,312.55,Delivered Late,Net Banking,285.0,False
13667,113668,10919.0,359.0,1132,2024-02-29,17:49:00,38.0,226,0,0.0,NaN,11.30,237.30,Delivered,Debit Card,537.0,False
13666,113667,984.0,147.0,1556,2024-03-26,18:01:00,24.0,569,49,170.7,ZOM153SL,28.45,475.75,Delivered,Wallet,278.0,False
13665,113666,6823.0,334.0,591,NaT,15:49:00,37.0,504,49,0.0,NaN,25.20,578.20,Delivered,Debit Card,133.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13701,113702,5790.0,1179.0,424,2023-12-25,19:00:00,27.0,348,40,0.0,NaN,17.40,405.40,Delivered,UPI,348.0,True
5354,105355,8311.0,939.0,631,2024-08-20,19:08:00,41.0,422,40,0.0,NaN,21.10,483.10,Delivered,UPI,422.0,True
7765,107766,84.0,191.0,1456,NaT,23:38:00,33.0,173,25,86.5,ZOM6SYNB,8.65,120.15,Delivered,Net Banking,173.0,True
16028,116029,1650.0,1143.0,1043,2024-03-05,14:45:00,23.0,295,25,0.0,NaN,14.75,334.75,Delivered,UPI,295.0,True


In [314]:
food_cost_check[food_cost_check['FoodCostMatch']==True]

,OrderID,CustomerID,RestaurantID,DeliveryPartnerID,OrderDate,OrderTime,DeliveryTimeMinutes,FoodCost,DeliveryFee,Discount,CouponCode,GST,FinalAmount,OrderStatus,PaymentMethod,OrderItemTotalPrice,FoodCostMatch
168,100169,11141.0,308.0,687,2024-10-27,18:24:00,39.0,402,0,40.2,ZOMBQZT6,20.10,381.90,Delivered,Net Banking,402.0,True
521,100522,6037.0,919.0,1043,2024-11-15,08:01:00,27.0,258,25,0.0,NaN,12.90,295.90,Delivered,Debit Card,258.0,True
3023,103024,8191.0,945.0,418,2023-01-19,11:02:00,30.0,308,40,92.4,ZOM7HZUQ,15.40,271.00,Food Not Delivered,Wallet,308.0,True
5354,105355,8311.0,939.0,631,2024-08-20,19:08:00,41.0,422,40,0.0,NaN,21.10,483.10,Delivered,UPI,422.0,True
5592,105593,7100.0,943.0,973,2023-10-02,12:42:00,36.0,420,0,0.0,NaN,21.00,NaN,Food Not Delivered,Net Banking,420.0,True
7765,107766,84.0,191.0,1456,NaT,23:38:00,33.0,173,25,86.5,ZOM6SYNB,8.65,120.15,Delivered,Net Banking,173.0,True
8629,108630,789.0,1164.0,251,2023-08-08,18:40:00,20.0,480,30,144.0,ZOM6C7AY,24.00,390.00,Delivered,Wallet,480.0,True
8775,108776,8359.0,733.0,1232,2023-03-22,08:15:00,32.0,357,0,0.0,NaN,17.85,374.85,Cancelled,Credit Card,357.0,True
11127,111128,11107.0,523.0,151,2023-04-12,10:59:00,40.0,200,25,40.0,ZOMHYX2Y,10.00,195.00,Food Not Delivered,Cash on Delivery,200.0,True
11655,111656,4656.0,266.0,968,2024-08-12,20:28:00,23.0,562,40,0.0,NaN,28.10,630.10,Delivered,Debit Card,562.0,True


In [315]:
food_cost_check['FoodCostMatch'].value_counts()

,count
FoodCostMatch,
False,20483
True,17


In [316]:
# checking for null entries in the FinalAmount column
(clean_datasets['orders']['FinalAmount'].isna()).value_counts()

,count
FinalAmount,
False,20175
True,325


In [317]:
# checking orders dataset with FinalAmount entries not as null
with_final_amount = clean_datasets['orders'].loc[clean_datasets['orders']['FinalAmount'].notna()]

In [318]:
with_final_amount.sort_values('FoodCost')

,OrderID,CustomerID,RestaurantID,DeliveryPartnerID,OrderDate,OrderTime,DeliveryTimeMinutes,FoodCost,DeliveryFee,Discount,CouponCode,GST,FinalAmount,OrderStatus,PaymentMethod
16478,116479,1050.0,749.0,1605,2023-07-22,12:05:00,37.0,-811,20,0.00,NaN,40.55,871.55,Delivered,Cash on Delivery
19903,119904,8723.0,85.0,203,2023-05-27,09:42:00,36.0,-807,20,0.00,NaN,40.35,867.35,Delivered,UPI
1774,101775,6635.0,1043.0,66,2024-05-14,19:20:00,43.0,-682,0,204.60,ZOM4QNB2,34.10,511.50,Cancelled,Credit Card
15746,115747,7176.0,335.0,1607,2023-10-12,15:22:00,10.0,-672,30,0.00,NaN,33.60,735.60,Delivered,UPI
9827,109828,8933.0,15.0,1428,2023-09-25,19:08:00,43.0,-671,0,0.00,NaN,33.55,704.55,Delivered,Cash on Delivery
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5670,105671,9835.0,556.0,1308,2023-12-18,14:48:00,48.0,921,0,0.00,NaN,46.05,967.05,Cancelled,Credit Card
4901,104902,5458.0,50.0,1288,2024-01-14,20:49:00,48.0,921,0,138.15,ZOMF62SP,46.05,828.90,Cancelled,Debit Card
6655,106656,5625.0,862.0,918,2024-11-22,12:19:00,44.0,944,20,0.00,NaN,47.20,1011.20,Cancelled,Cash on Delivery
4262,104263,3877.0,79.0,677,2023-04-26,10:33:00,17.0,949,40,379.60,ZOMNGA1X,47.45,656.85,Food Not Delivered,Cash on Delivery


In [319]:
# checking if the FinalAmount values match with the calculated FoodCost + DeliveryFee + GST - Discount
(with_final_amount['FinalAmount'] == (with_final_amount['FoodCost'] + with_final_amount['DeliveryFee'] + with_final_amount['GST'] - with_final_amount['Discount']).round(2)).value_counts()

,count
True,19997
False,178


In [320]:
# expected FinalAmount calculation
with_final_amount['ExpectedFinalAmount'] = (
    with_final_amount['FoodCost'].abs()
    + with_final_amount['DeliveryFee']
    + with_final_amount['GST']
    - with_final_amount['Discount']
)

/tmp/ipykernel_33721/3235354722.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  with_final_amount['ExpectedFinalAmount'] = (


In [321]:
# difference in between actual FinalAmount and ExpectedFinalAmount
with_final_amount['AmountDifference'] = (
    with_final_amount['FinalAmount']
    - with_final_amount['ExpectedFinalAmount']
)

/tmp/ipykernel_33721/1736990730.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  with_final_amount['AmountDifference'] = (


In [322]:
with_final_amount['AmountDifference'].describe()

,AmountDifference
count,2.017500e+04
mean,-1.458065e-16
std,1.758146e-14
min,-1.136868e-13
25%,0.000000e+00
50%,0.000000e+00
75%,0.000000e+00
max,1.136868e-13


In [323]:
# checking if there are any entries with AmountDifference greater than 0.01
invalid_final_amount = with_final_amount[
    with_final_amount['AmountDifference'].abs() > 0.01
]

invalid_final_amount[
    [
        'OrderID',
        'FoodCost',
        'DeliveryFee',
        'GST',
        'Discount',
        'FinalAmount',
        'ExpectedFinalAmount',
        'AmountDifference'
    ]
]

,OrderID,FoodCost,DeliveryFee,GST,Discount,FinalAmount,ExpectedFinalAmount,AmountDifference


In [324]:
# converting all the FoodCost entries to be in absolute format
clean_datasets['orders'].loc[clean_datasets['orders']['FinalAmount'].notna(),'FoodCost'] = clean_datasets['orders']['FoodCost'].abs()

In [325]:
(clean_datasets['orders']['FoodCost']<0).value_counts()

,count
FoodCost,
False,20497
True,3


In [326]:
clean_datasets['orders'].loc[clean_datasets['orders']['FoodCost']<0]

,OrderID,CustomerID,RestaurantID,DeliveryPartnerID,OrderDate,OrderTime,DeliveryTimeMinutes,FoodCost,DeliveryFee,Discount,CouponCode,GST,FinalAmount,OrderStatus,PaymentMethod
2131,102132,8374.0,540.0,4,NaT,08:13:00,32.0,-606,40,0.0,NaN,30.3,NaN,Food Not Delivered,Net Banking
19658,119659,318.0,45.0,745,2024-11-24,15:56:00,85.0,-476,30,95.2,ZOMVX6UU,23.8,NaN,Delivered Late,UPI
19877,119878,4446.0,911.0,745,2024-05-12,12:09:00,NaN,-324,0,0.0,NaN,16.2,NaN,Delivered,Cash on Delivery


In [327]:
cleaning_decisions(
    dataset = 'orders',
    column = 'FoodCost',
    issue = 'negative FoodCost entries',
    evidence = 'there are 181 negative records in FoodCost columns',
    action = 'converted the negative records to abs where we have FinalAmount value not as NA',
    reason = 'aligns with the formula FinalAmount = FoodCost + DeliveryFee + GST - Discount',
    before_count = '181',
    changes_count = '178',
    after_count = '3',
    status = 'Completed'
)

In [328]:
clean_datasets['orders'].loc[clean_datasets['orders']['FoodCost']<0,'FoodCost'] = pd.NA

In [329]:
valid_amount = (
    clean_datasets['orders']['FoodCost'].notna()
    & clean_datasets['orders']['DeliveryFee'].notna()
    & clean_datasets['orders']['GST'].notna()
    & clean_datasets['orders']['Discount'].notna()
)

clean_datasets['orders'].loc[valid_amount, 'FinalAmount'] = (
    clean_datasets['orders'].loc[valid_amount, 'FoodCost']
    + clean_datasets['orders'].loc[valid_amount, 'DeliveryFee']
    + clean_datasets['orders'].loc[valid_amount, 'GST']
    - clean_datasets['orders'].loc[valid_amount, 'Discount']
).round(2)

In [330]:
clean_datasets['orders']['FinalAmount'].notna().value_counts()

,count
FinalAmount,
True,20497
False,3


In [331]:
cleaning_decisions(
    dataset = 'orders',
    column = 'FinalAmount',
    issue = 'missing values',
    evidence = 'there are 325 missing entries in the FinalAmount column',
    action = 'calculated the missing FinalAmount values based on other entries',
    reason = 'missing values with evidence can be filled',
    before_count = '325',
    changes_count = '322',
    after_count = '3',
    status = 'Completed'
)

In [332]:
clean_datasets['orders']['OrderStatus'].value_counts()

,count
OrderStatus,
Delivered,11407
Cancelled,4573
Food Not Delivered,2364
Delivered Late,2156


In [333]:
clean_datasets['orders']['PaymentMethod'].value_counts()

,count
PaymentMethod,
Debit Card,3537
Net Banking,3464
Cash on Delivery,3445
UPI,3353
Wallet,3351
Credit Card,3350


In [334]:
# checking CouponCode integrity with promotions dataset
(clean_datasets['orders']['CouponCode'].notna().isin(clean_datasets['promotions']['CouponCode'])).value_counts()

,count
CouponCode,
False,20500


In [335]:
dataset_profiling(clean_datasets['orders'])

Rows: 20500
Columns: 15
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,OrderID,0,0,int64,20500
1,CustomerID,12,10670,float64,9829
2,RestaurantID,13,19299,float64,1200
3,DeliveryPartnerID,0,18500,int64,2000
4,OrderDate,2913,19768,datetime64[ns],731
5,OrderTime,0,19075,object,1425
6,DeliveryTimeMinutes,353,20403,float64,96
7,FoodCost,3,19702,float64,797
8,DeliveryFee,0,20494,int64,6
9,Discount,0,18337,float64,2163


In [336]:
column_description(clean_datasets['orders'])

Column: OrderID
**************************************************
Description:

count     20500.000000
mean     110250.500000
std        5917.984595
min      100001.000000
25%      105125.750000
50%      110250.500000
75%      115375.250000
max      120500.000000
Name: OrderID, dtype: float64
Column: CustomerID
**************************************************
Description:

count    20488.000000
mean      5965.707634
std       3482.505387
min          1.000000
25%       2913.750000
50%       5905.000000
75%       9022.000000
max      11999.000000
Name: CustomerID, dtype: float64
Column: RestaurantID
**************************************************
Description:

count    20487.000000
mean       602.445307
std        347.868079
min          1.000000
25%        303.000000
50%        600.000000
75%        904.000000
max       1200.000000
Name: RestaurantID, dtype: float64
Column: DeliveryPartnerID
**************************************************
Description:

count    20500.000000
me

In [337]:
dataset_profiling(clean_datasets['orders'])

Rows: 20500
Columns: 15
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,OrderID,0,0,int64,20500
1,CustomerID,12,10670,float64,9829
2,RestaurantID,13,19299,float64,1200
3,DeliveryPartnerID,0,18500,int64,2000
4,OrderDate,2913,19768,datetime64[ns],731
5,OrderTime,0,19075,object,1425
6,DeliveryTimeMinutes,353,20403,float64,96
7,FoodCost,3,19702,float64,797
8,DeliveryFee,0,20494,int64,6
9,Discount,0,18337,float64,2163


In [338]:
payments_date = (
    clean_datasets['payments']
    .groupby('OrderID', as_index=False)['PaymentDate']
    .first()
)

In [339]:
payments_date.head()

,OrderID,PaymentDate
0,100001,2023-06-17
1,100002,2023-07-01
2,100003,2023-05-09
3,100004,2024-06-22
4,100005,2024-03-17


In [340]:
orders_payments = clean_datasets['orders'][['OrderID','OrderDate']].merge(
    payments_date,
    on = 'OrderID',
    how = 'right'
)

In [341]:
orders_payments.head()

,OrderID,OrderDate,PaymentDate
0,100001,2023-06-17,2023-06-17
1,100002,2023-07-01,2023-07-01
2,100003,2023-05-09,2023-05-09
3,100004,2024-06-22,2024-06-22
4,100005,2024-03-17,2024-03-17


In [342]:
orders_payments['date_status'] = 'Match'

orders_payments.loc[
    orders_payments['PaymentDate'].isna(),
    'date_status'
] = 'No Payment'

orders_payments.loc[
    orders_payments['PaymentDate'].notna() &
    (orders_payments['OrderDate'] != orders_payments['PaymentDate']),
    'date_status'
] = 'Date Mismatch'

In [343]:
orders_payments['date_status'].value_counts()

,count
date_status,
Match,15522
No Payment,2835
Date Mismatch,1737


In [344]:
date_mismatches = orders_payments[
    orders_payments['date_status'] == 'Date Mismatch'
].sort_values('OrderDate')

In [345]:
date_mismatches

,OrderID,OrderDate,PaymentDate,date_status
6,100007,NaT,2024-08-10,Date Mismatch
13,100014,NaT,2023-03-10,Date Mismatch
24,100025,NaT,2024-03-01,Date Mismatch
31,100032,NaT,2023-07-03,Date Mismatch
36,100038,NaT,2023-05-09,Date Mismatch
...,...,...,...,...
20049,120455,NaT,2023-05-01,Date Mismatch
20056,120462,NaT,2023-12-07,Date Mismatch
20057,120463,NaT,2023-10-09,Date Mismatch
20062,120468,NaT,2024-07-03,Date Mismatch


In [346]:
date_adding_from_payments_orders = date_mismatches[
    ['OrderID', 'PaymentDate']
]

payment_dates = date_adding_from_payments_orders.set_index(
    'OrderID'
)['PaymentDate']

clean_datasets['orders']['OrderDate'] = (
    clean_datasets['orders']['OrderID']
    .map(payment_dates)
    .fillna(clean_datasets['orders']['OrderDate'])
)

In [347]:
cleaning_decisions(
    dataset = 'orders',
    column = 'OrderDate',
    issue = 'missing/converted NaT values',
    evidence = 'there are NaT values in the orders table that can be replaced with the help of payments table',
    action = 'replaced 1737 values in the OrderDate column of the orders dataset',
    reason = 'missing values with truth found in other datasets can be replaced',
    before_count = '2913',
    changes_count = '1737',
    after_count = '1176',
    status = 'Completed'
)

cities dataset cleaning decisions

In [348]:
dataset_profiling(clean_datasets['cities'])

Rows: 25
Columns: 5
**************************************************


,Column,Missing,Duplicate,Dtype,unique
0,CityID,0,0,int64,25
1,City,0,0,object,25
2,Population,0,0,int64,25
3,Region,0,21,object,4
4,AverageIncome,2,1,float64,23


In [349]:
cleaning_decisions(
    dataset = 'cities',
    column = 'AverageIncome',
    issue = 'missing AverageIncome entries',
    evidence = 'there are 2 entries with missing AverageIncome',
    action = 'retained missing',
    reason = 'cannot fabricate city data where multiple parameters are involved',
    before_count = '2',
    changes_count = '0',
    after_count = '2',
    status = 'Profiled'
)

In [350]:
column_description(clean_datasets['cities'])

Column: CityID
**************************************************
Description:

count    25.000000
mean     13.000000
std       7.359801
min       1.000000
25%       7.000000
50%      13.000000
75%      19.000000
max      25.000000
Name: CityID, dtype: float64
Column: City
**************************************************
Description:

count         25
unique        25
top       Mumbai
freq           1
Name: City, dtype: object
Column: Population
**************************************************
Description:

count    2.500000e+01
mean     6.594987e+06
std      7.486161e+06
min      1.059078e+06
25%      2.177335e+06
50%      3.043446e+06
75%      8.636140e+06
max      3.293902e+07
Name: Population, dtype: float64
Column: Region
**************************************************
Description:

count       25
unique       4
top       West
freq         9
Name: Region, dtype: object
Column: AverageIncome
**************************************************
Description:

count    2.300000e+

In [351]:
# checking unique entries in Region column of the cities dataset
clean_datasets['cities']['Region'].value_counts()

,count
Region,
West,9
South,6
North,5
East,5


In [352]:
# saving cleaned data
cleaned_path = '/content/drive/MyDrive/Internmo/Zomato1/cleaned_data'

os.makedirs(cleaned_path, exist_ok=True)

In [353]:
for name, df in clean_datasets.items():
    file_path = os.path.join(cleaned_path, f'{name}.csv')
    df.to_csv(file_path, index=False)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!


In [354]:
saved_files = glob.glob(cleaned_path + '/*.csv')

print(f"Number of files saved: {len(saved_files)}")

for file in saved_files:
    print(file)

Number of files saved: 12
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/customer_feedback.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/traffic.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/weather.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/order_items.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/payments.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/menu.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/delivery_partners.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/promotions.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/restaurants.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/customers.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/orders.csv
/content/drive/MyDrive/Internmo/Zomato1/cleaned_data/cities.csv


In [355]:
for name in clean_datasets:
    file_path = os.path.join(cleaned_path, f'{name}.csv')

    test_df = pd.read_csv(file_path)

    print(
        f"{name}: "
        f"{test_df.shape[0]} rows × {test_df.shape[1]} columns"
    )

customer_feedback: 13970 rows × 7 columns
traffic: 18335 rows × 6 columns
weather: 18173 rows × 7 columns
order_items: 42378 rows × 6 columns
payments: 20291 rows × 6 columns
menu: 8997 rows × 8 columns
delivery_partners: 2000 rows × 10 columns
promotions: 300 rows × 6 columns
restaurants: 1200 rows × 13 columns
customers: 12000 rows × 15 columns
orders: 20500 rows × 15 columns
cities: 25 rows × 5 columns


In [356]:
clean_colum_profiling = basic_columns_profiling(clean_datasets)
clean_colum_profiling.to_csv('/content/drive/MyDrive/Internmo/Zomato1/clean_column_profile.csv',index=False)

In [357]:
cleaning_decisions_profiling = pd.DataFrame(cleaning_log)
cleaning_decisions_profiling.to_csv('/content/drive/MyDrive/Internmo/Zomato1/cleaning_decisions_Profiling.csv',index=False)

In [358]:
# Folder in Google Drive where MySQL-ready CSVs will be saved
output_dir = "/content/drive/MyDrive/Internmo/Zomato1/mysql_ready_csv"
os.makedirs(output_dir, exist_ok=True)

# Date columns that should become YYYY-MM-DD
date_cols = {
    "customers": ["RegistrationDate"],
    "delivery_partners": ["JoiningDate"],
    "promotions": ["StartDate", "EndDate"],
    "orders": ["OrderDate"],
    "payments": ["PaymentDate"],
    "weather": ["Date"],
    "traffic": ["Date"]
}

# Time columns that should become HH:MM:SS
time_cols = {
    "orders": ["OrderTime"],
    "traffic": ["Time"]
}

for name, df in clean_datasets.items():

    out = df.copy()

    # Convert DATE columns
    for col in date_cols.get(name, []):
        if col in out.columns:
            out[col] = pd.to_datetime(
                out[col], errors="coerce"
            ).dt.strftime("%Y-%m-%d")

    # Convert TIME columns
    for col in time_cols.get(name, []):
        if col in out.columns:
            # Convert to time format
            converted = pd.to_datetime(
                out[col].astype(str),
                errors="coerce"
            )

            out[col] = converted.dt.strftime("%H:%M:%S")

    # Save:
    # - index=False → don't create an extra pandas index column
    # - na_rep='\\N' → MySQL LOAD DATA interprets this as NULL
    output_path = os.path.join(output_dir, f"{name}.csv")

    out.to_csv(
        output_path,
        index=False,
        na_rep="\\N"
    )

    print(f"Saved: {name}.csv | Rows: {len(out):,}")

print("\nAll MySQL-ready CSVs saved to:")
print(output_dir)

Saved: customer_feedback.csv | Rows: 13,970


/tmp/ipykernel_33721/3595338985.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(


Saved: traffic.csv | Rows: 18,335
Saved: weather.csv | Rows: 18,173
Saved: order_items.csv | Rows: 42,378
Saved: payments.csv | Rows: 20,291
Saved: menu.csv | Rows: 8,997
Saved: delivery_partners.csv | Rows: 2,000
Saved: promotions.csv | Rows: 300
Saved: restaurants.csv | Rows: 1,200
Saved: customers.csv | Rows: 12,000


/tmp/ipykernel_33721/3595338985.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(


Saved: orders.csv | Rows: 20,500
Saved: cities.csv | Rows: 25

All MySQL-ready CSVs saved to:
/content/drive/MyDrive/Internmo/Zomato1/mysql_ready_csv


In [359]:
files = sorted(os.listdir(output_dir))

print("Number of CSV files:", len(files))
print("\nFiles:")

for file in files:
    print(file)

Number of CSV files: 12

Files:
cities.csv
customer_feedback.csv
customers.csv
delivery_partners.csv
menu.csv
order_items.csv
orders.csv
payments.csv
promotions.csv
restaurants.csv
traffic.csv
weather.csv
